# Arquitetura campeã para Detecção de Anomalias Acústicas

Este notebook é uma versão **limpa, didática e focada apenas na arquitetura campeã** do projeto, agora com a ingestão que você usa na prática:

1. **seus áudios próprios no Google Drive**;
2. **Kaggle Bearing Dataset**, no mesmo padrão do notebook anterior (`archive.zip` ou pasta `kaggle_data/Dataset`);
3. **DCASE 2025 Task 2**, para alinhamento de protocolo e validação externa em `first-shot unsupervised anomalous sound detection`;
4. **MIMII**, em amostra pequena, quando houver pasta local ou quando você habilitar manualmente o download.

O objetivo não é repetir o benchmark gigante. Aqui usamos somente a arquitetura campeã:

- **Supervisionado:** `XGBoost + regularização L2`, com `Mixup` e busca leve de hiperparâmetros.
- **Não supervisionado:** `Mahalanobis + Distribuição Gamma`, com limiar controlado por FPR.
- **DSP:** `HHT + UKF` para limpeza/isolamento temporal.
- **RPCA:** separação robusta do espectrograma em padrão estável e componente esparsa de impacto.
- **Representação:** `Super-Vector` com MFCC, Mel, estatísticas físicas, descritores RPCA, erro NMF e agregação por arquivo.
- **Desafiantes resgatados da Frente 11:** `Isolation Forest`, `Tiny-AST` apenas para embeddings/XAI e `heatmaps de saliência` por gradiente.

## O que este notebook entrega

- Tabelas de ingestão por fonte: Drive, Kaggle, DCASE e MIMII, com auditoria de papéis de treino/teste/OE.
- Gráficos didáticos de distribuição dos dados.
- Resultados separados para o modelo supervisionado e o não supervisionado.
- Teste cego para mudança de domínio usando a pasta própria de teste do Drive.
- Métricas industriais: `F1`, `AUC`, `pAUC@0.1`, matriz de confusão, ROC, PR e latência.
- XAI visual com waveform, espectrograma, mapa esparso RPCA, hotspots, ativação por banda e explicações em linguagem simples.
- Relatório final comparando os vencedores do DCASE 2025, o que eles usaram e o que a nossa arquitetura usou.

> Regra de segurança metodológica: dados de teste cego e dados externos de avaliação não entram no treino. Eles são usados apenas para medir generalização e mudança de domínio.

## Da Frente 11 para o produto

A Frente 11 era uma arena de benchmarking amplo, com muitos modelos disputando desempenho. A Arquitetura Campeã reduz esse universo para um pipeline mais enxuto, auditável e pronto para Edge. Nesta versão, o núcleo operacional continua sendo `DSP + Super-Vector + XGBoost/Mahalanobis`, mas os blocos que ajudavam a investigar `domain shift` na Frente 11 foram resgatados como trilhas opcionais: `Isolation Forest` para caos estatístico, `Tiny-AST` apenas como extrator de embeddings/reconstrução e `saliência` visual para vender o diagnóstico com mais força.

## Mapa didático da arquitetura campeã

Pense no notebook como uma linha de produção. O áudio entra cru, passa por filtros, vira números e depois os modelos decidem se o som parece normal ou suspeito.

| Etapa | Nome técnico | Explicação para leigos | Está no notebook? |
|---|---|---|---|
| 1 | Janelas deslizantes | O áudio é cortado em pedaços pequenos para o modelo não analisar tudo de uma vez. | Sim |
| 2 | HHT + UKF | Limpa o sinal e evidencia batidas, atritos e vibrações importantes. | Sim |
| 3 | RPCA | Separa o som repetitivo da máquina dos eventos raros, como impactos. | Sim |
| 4 | MFCC, Mel e estatísticas | Transforma som em números que descrevem timbre, energia, rugosidade e impulsividade. | Sim |
| 5 | NMF | Aprende a receita do som normal e mede o quanto um novo som foge dela. | Sim |
| 6 | Mixup | Cria exemplos intermediários entre normal e falha; por padrão usa `lambda=0.35`. Também gera pseudo-normais entre domínios diferentes. | Sim, controlado por parâmetro |
| 7 | OE | Usa sons auxiliares externos como exemplos de fora do padrão para endurecer a fronteira. | Opcional e auditado |
| 8 | XGBoost + L2 | Modelo supervisionado quando existem exemplos rotulados de falha. | Sim |
| 9 | Mahalanobis + Gamma | Modelo não supervisionado quando só existem sons normais. | Sim |
| 9b | GMM + Gamma | Desafiante não supervisionado para verificar se a normalidade tem múltiplos modos. | Sim, investigativo |
| 9c | Isolation Forest | Desafiante não supervisionado baseado em árvores para ambientes mais caóticos. | Sim, investigativo |
| 9d | Tiny-AST Embeddings | Extrator auxiliar de embeddings profundos e reconstrutor visual, sem substituir o campeão leve. | Sim, opcional |
| 10 | pAUC@0.1 | Mede desempenho na região de poucos falsos alarmes, a mais importante para indústria. | Sim |
| 11 | Teste cego | Mede mudança de domínio: outro microfone, outro ambiente, outra máquina. | Sim |
| 12 | XAI | Mostra onde e por que o modelo encontrou evidência de anomalia. | Sim |

**Nota importante:** a Frente 11 original era um laboratório de busca. Este notebook fixa a arquitetura campeã, mas agora mantém os principais mecanismos aplicados/avaliados: Mixup, Gamma, pAUC, RPCA, NMF, teste cego, comparação DCASE, OE opcional, IForest, Tiny-AST auxiliar e saliência visual. Assim ele fica mais fácil de explicar e reproduzir.


In [ ]:
# ============================================================
# 1. Instalação segura das dependências
# ============================================================
# Em Colab, a primeira execução pode demorar alguns minutos.

import sys
import subprocess
import importlib.util

PACKAGE_IMPORTS = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'librosa': 'librosa',
    'soundfile': 'soundfile',
    'tqdm': 'tqdm',
    'joblib': 'joblib',
    'requests': 'requests',
    'remotezip': 'remotezip',
    'bs4': 'beautifulsoup4',
    'filterpy': 'filterpy',
    'xgboost': 'xgboost',
    'torch': 'torch',
    'PyEMD': 'EMD-signal',
}

missing = []
for import_name, pip_name in PACKAGE_IMPORTS.items():
    if importlib.util.find_spec(import_name) is None:
        missing.append(pip_name)

if missing:
    print('Instalando:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('Dependências principais já instaladas.')


In [ ]:
# ============================================================
# 2. Imports e estilo visual
# ============================================================

from __future__ import annotations

import os
import json
import time
import zipfile
import warnings
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import librosa
import librosa.display
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from scipy import signal as scipy_signal
from scipy import stats
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)
from sklearn.decomposition import NMF
from sklearn.mixture import GaussianMixture
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
import joblib

warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    XGBClassifier = None
    HAS_XGBOOST = False

try:
    from filterpy.kalman import UnscentedKalmanFilter as UKF
    from filterpy.kalman import MerweScaledSigmaPoints
    HAS_FILTERPY = True
except Exception:
    HAS_FILTERPY = False

try:
    from PyEMD import EMD
    HAS_PYEMD = True
except Exception:
    HAS_PYEMD = False

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except Exception:
    torch = None
    nn = None
    optim = None
    DataLoader = None
    TensorDataset = None
    HAS_TORCH = False

sns.set_theme(style='whitegrid')
sns.set_context('talk', font_scale=1.0)
plt.rcParams.update({
    'figure.figsize': (13, 6),
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'axes.titlesize': 17,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print('XGBoost:', HAS_XGBOOST)
print('FilterPy/UKF:', HAS_FILTERPY)
print('PyEMD/HHT real:', HAS_PYEMD)
print('PyTorch/Tiny-AST auxiliar:', HAS_TORCH)

In [ ]:
# ============================================================
# 3. Configuração de caminhos, Drive, fontes e limites
# ============================================================
# Esta célula replica a lógica do notebook anterior: no Colab, monta o Drive;
# localmente, usa a pasta do projeto. Também mantém o Kaggle no padrão
# `archive.zip` ou `kaggle_data/Dataset`.

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    if os.getenv('AAD_MOUNT_DRIVE', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}:
        drive.mount('/content/drive', force_remount=False)
    DEFAULT_DRIVE_AUDIO_ROOT = Path('/content/drive/MyDrive/PROJETO APLICADO - Acoustic Anomaly Detection (AAD)/Arquivos de audio')
    PROJECT_ROOT = Path('/content/aad_arquitetura_campea')
else:
    PROJECT_ROOT = Path('/Users/emanoelspanhol/Downloads/V2 benchmarking 2')
    DEFAULT_DRIVE_AUDIO_ROOT = PROJECT_ROOT / 'PROJETO APLICADO - Acoustic Anomaly Detection (AAD)' / 'Arquivos de audio'
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_AUDIO_ROOT = Path(os.getenv('AAD_DRIVE_AUDIO_ROOT', str(DEFAULT_DRIVE_AUDIO_ROOT))).expanduser()


def parse_path_list(value: str) -> list[Path]:
    return [Path(item.strip()).expanduser() for item in value.split(',') if item.strip()]


def existing_or_default_paths(env_name: str, defaults: list[Path]) -> list[Path]:
    raw = os.getenv(env_name, '').strip()
    if raw:
        return parse_path_list(raw)
    return defaults

OWN_DATA_DIRS = existing_or_default_paths('AAD_OWN_DATA_DIRS', [
    DRIVE_AUDIO_ROOT / 'audios_proprios',
    DRIVE_AUDIO_ROOT / 'proprios',
    DRIVE_AUDIO_ROOT / 'meus_audios',
])

BLIND_TEST_DIRS = existing_or_default_paths('AAD_BLIND_TEST_DIRS', [
    DRIVE_AUDIO_ROOT / 'teste',
    DRIVE_AUDIO_ROOT / 'teste_cego',
])

DATASETS_ROOT = Path(os.getenv('AAD_DATASETS_ROOT', str(PROJECT_ROOT / 'datasets'))).expanduser()
DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

KAGGLE_ZIP = Path(os.getenv('AAD_KAGGLE_ZIP', str(DRIVE_AUDIO_ROOT / 'archive.zip'))).expanduser()
KAGGLE_EXTRACT_ROOT = Path(os.getenv('AAD_KAGGLE_EXTRACT_ROOT', '/content/kaggle_data' if IN_COLAB else str(DRIVE_AUDIO_ROOT / 'kaggle_data'))).expanduser()
KAGGLE_ROOT = Path(os.getenv('AAD_KAGGLE_ROOT', str(DRIVE_AUDIO_ROOT / 'kaggle_data' / 'Dataset'))).expanduser()

DCASE_RECORD_ID = '15097779'
DCASE_EXTRACT_ROOT = DATASETS_ROOT / 'dcase2025_task2' / 'dev_data'
DCASE_RAW_ROOT = DCASE_EXTRACT_ROOT / 'raw'
DCASE_DOWNLOAD_ROOT = DATASETS_ROOT / 'dcase2025_task2' / 'zips'
DCASE_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
DCASE_RAW_ROOT.mkdir(parents=True, exist_ok=True)
DCASE_PACKAGES = [p.strip() for p in os.getenv('AAD_DCASE_PACKAGES', 'dev_valve.zip,dev_bearing.zip').split(',') if p.strip()]
AUTO_DOWNLOAD_DCASE = os.getenv('AAD_AUTO_DOWNLOAD_DCASE', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}

MIMII_RECORD_ID = '3384388'
MIMII_ROOT = Path(os.getenv('AAD_MIMII_ROOT', str(DATASETS_ROOT / 'mimii'))).expanduser()
ENABLE_MIMII_SCAN = os.getenv('AAD_ENABLE_MIMII_SCAN', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
AUTO_DOWNLOAD_MIMII = os.getenv('AAD_AUTO_DOWNLOAD_MIMII', '0').strip().lower() in {'1', 'true', 'yes', 'sim'}
MIMII_PACKAGES = [p.strip() for p in os.getenv('AAD_MIMII_PACKAGES', '6_dB_valve.zip').split(',') if p.strip()]
MIMII_RECOMMENDED_PACKAGE = os.getenv('AAD_MIMII_RECOMMENDED_PACKAGE', '6_dB_valve.zip').strip()
MIMII_REMOTE_SAMPLE_ENABLED = os.getenv('AAD_MIMII_REMOTE_SAMPLE_ENABLED', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
MIMII_REMOTE_SAMPLE_SIZE = int(os.getenv('AAD_MIMII_REMOTE_SAMPLE_SIZE', '500'))

SR_TARGET = int(os.getenv('AAD_SR_TARGET', '16000'))
CSV_SIGNAL_SR = int(os.getenv('AAD_CSV_SIGNAL_SR', '16000'))
WINDOW_SEC = float(os.getenv('AAD_WINDOW_SEC', '1.0'))
HOP_SEC = float(os.getenv('AAD_HOP_SEC', '0.50'))
MAX_WINDOWS_PER_FILE = int(os.getenv('AAD_MAX_WINDOWS_PER_FILE', '10'))
MAX_OWN_PER_LABEL = int(os.getenv('AAD_MAX_OWN_PER_LABEL', '80'))
MAX_BLIND_FILES = int(os.getenv('AAD_MAX_BLIND_FILES', '80'))
MAX_KAGGLE_PER_CLASS = int(os.getenv('AAD_MAX_KAGGLE_PER_CLASS', '120'))
MAX_DCASE_TRAIN_PER_MACHINE = int(os.getenv('AAD_MAX_DCASE_TRAIN_PER_MACHINE', '90'))
MAX_DCASE_TEST_PER_MACHINE_LABEL = int(os.getenv('AAD_MAX_DCASE_TEST_PER_MACHINE_LABEL', '60'))
MAX_MIMII_PER_LABEL = int(os.getenv('AAD_MAX_MIMII_PER_LABEL', '250'))
MAX_MIMII_TOTAL_FILES = int(os.getenv('AAD_MAX_MIMII_TOTAL_FILES', '500'))
FPR_TARGET = float(os.getenv('AAD_FPR_TARGET', '0.10'))
MIXUP_ENABLED = os.getenv('AAD_MIXUP_ENABLED', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
MIXUP_RATIO = float(os.getenv('AAD_MIXUP_RATIO', '0.25'))
MIXUP_USE_FIXED_LAMBDA = os.getenv('AAD_MIXUP_USE_FIXED_LAMBDA', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
MIXUP_LAMBDA = float(os.getenv('AAD_MIXUP_LAMBDA', '0.35'))
MIXUP_ALPHA = float(os.getenv('AAD_MIXUP_ALPHA', '0.40'))
MIXUP_MAX_SAMPLES = int(os.getenv('AAD_MIXUP_MAX_SAMPLES', '800'))
MIXUP_SAMPLE_WEIGHT = float(os.getenv('AAD_MIXUP_SAMPLE_WEIGHT', '0.55'))
DOMAIN_MIXUP_ENABLED = os.getenv('AAD_DOMAIN_MIXUP_ENABLED', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
DOMAIN_MIXUP_RATIO = float(os.getenv('AAD_DOMAIN_MIXUP_RATIO', '0.25'))
DOMAIN_MIXUP_SAMPLE_WEIGHT = float(os.getenv('AAD_DOMAIN_MIXUP_SAMPLE_WEIGHT', '0.45'))
DOMAIN_MIXUP_USE_UNSUP_POOL = os.getenv('AAD_DOMAIN_MIXUP_USE_UNSUP_POOL', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
DOMAIN_MIXUP_PREFER_SOURCE_SHIFT = os.getenv('AAD_DOMAIN_MIXUP_PREFER_SOURCE_SHIFT', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
OE_ENABLED = os.getenv('AAD_OE_ENABLED', '0').strip().lower() in {'1', 'true', 'yes', 'sim'}
OE_MAX_WINDOWS = int(os.getenv('AAD_OE_MAX_WINDOWS', '500'))
OE_SAMPLE_WEIGHT = float(os.getenv('AAD_OE_SAMPLE_WEIGHT', '0.35'))
ENABLE_HYPERPARAM_SEARCH = os.getenv('AAD_ENABLE_HYPERPARAM_SEARCH', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
HYPERPARAM_RANKING_METRIC = os.getenv('AAD_HYPERPARAM_RANKING_METRIC', 'pauc_0_1').strip()
ENABLE_GMM_CHALLENGER = os.getenv('AAD_ENABLE_GMM_CHALLENGER', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
GMM_MAX_COMPONENTS = int(os.getenv('AAD_GMM_MAX_COMPONENTS', '4'))
GMM_COVARIANCE_TYPE = os.getenv('AAD_GMM_COVARIANCE_TYPE', 'full').strip()
ENABLE_IFOREST_CHALLENGER = os.getenv('AAD_ENABLE_IFOREST_CHALLENGER', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
IFOREST_N_ESTIMATORS = int(os.getenv('AAD_IFOREST_N_ESTIMATORS', '300'))
IFOREST_MAX_SAMPLES = os.getenv('AAD_IFOREST_MAX_SAMPLES', 'auto').strip()
IFOREST_BOOTSTRAP = os.getenv('AAD_IFOREST_BOOTSTRAP', '0').strip().lower() in {'1', 'true', 'yes', 'sim'}
ENABLE_TINY_AST_AUX = os.getenv('AAD_ENABLE_TINY_AST_AUX', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
AST_IMAGE_MELS = int(os.getenv('AAD_AST_IMAGE_MELS', '64'))
AST_IMAGE_TIME_BINS = int(os.getenv('AAD_AST_IMAGE_TIME_BINS', '64'))
TINY_AST_EMBED_DIM = int(os.getenv('AAD_TINY_AST_EMBED_DIM', '48'))
TINY_AST_HEADS = int(os.getenv('AAD_TINY_AST_HEADS', '4'))
TINY_AST_LAYERS = int(os.getenv('AAD_TINY_AST_LAYERS', '2'))
TINY_AST_EPOCHS = int(os.getenv('AAD_TINY_AST_EPOCHS', '4'))
TINY_AST_BATCH_SIZE = int(os.getenv('AAD_TINY_AST_BATCH_SIZE', '32'))
TINY_AST_MAX_TRAIN_WINDOWS = int(os.getenv('AAD_TINY_AST_MAX_TRAIN_WINDOWS', '1200'))
TINY_AST_LR = float(os.getenv('AAD_TINY_AST_LR', '1e-3'))
TINY_AST_ENABLE_SALIENCY = os.getenv('AAD_TINY_AST_ENABLE_SALIENCY', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
INCLUDE_KAGGLE_BLIND = os.getenv('AAD_INCLUDE_KAGGLE_BLIND', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
INCLUDE_DCASE_BLIND = os.getenv('AAD_INCLUDE_DCASE_BLIND', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
INCLUDE_MIMII_BLIND = os.getenv('AAD_INCLUDE_MIMII_BLIND', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
MAX_KAGGLE_BLIND_PER_CLASS = int(os.getenv('AAD_MAX_KAGGLE_BLIND_PER_CLASS', '16'))
MAX_DCASE_BLIND_PER_MACHINE_LABEL = int(os.getenv('AAD_MAX_DCASE_BLIND_PER_MACHINE_LABEL', '16'))
MAX_MIMII_BLIND_PER_LABEL = int(os.getenv('AAD_MAX_MIMII_BLIND_PER_LABEL', '20'))

ARTIFACT_DIR = PROJECT_ROOT / 'artifacts_campea'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def kaggle_root_is_valid(root: Path) -> bool:
    root = Path(root)
    return root.exists() and all((root / folder).exists() for folder in ['Normal', 'Inner Race Fault', 'Outer Race Fault'])

if not kaggle_root_is_valid(KAGGLE_ROOT):
    candidate = KAGGLE_EXTRACT_ROOT / 'Dataset'
    if kaggle_root_is_valid(candidate):
        KAGGLE_ROOT = candidate
    elif KAGGLE_ZIP.exists():
        print('Extraindo Kaggle archive.zip...')
        KAGGLE_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(KAGGLE_ZIP, 'r') as zf:
            zf.extractall(KAGGLE_EXTRACT_ROOT)
        if kaggle_root_is_valid(candidate):
            KAGGLE_ROOT = candidate
        elif kaggle_root_is_valid(KAGGLE_EXTRACT_ROOT):
            KAGGLE_ROOT = KAGGLE_EXTRACT_ROOT

config_df = pd.DataFrame([
    ['Ambiente', 'Colab' if IN_COLAB else 'Local', ''],
    ['Raiz Drive/áudios próprios', str(DRIVE_AUDIO_ROOT), DRIVE_AUDIO_ROOT.exists()],
    ['Pastas próprias', '\n'.join(str(p) for p in OWN_DATA_DIRS), ''],
    ['Pastas teste cego', '\n'.join(str(p) for p in BLIND_TEST_DIRS), ''],
    ['Kaggle root', str(KAGGLE_ROOT), kaggle_root_is_valid(KAGGLE_ROOT)],
    ['Kaggle zip', str(KAGGLE_ZIP), KAGGLE_ZIP.exists()],
    ['DCASE raw', str(DCASE_RAW_ROOT), DCASE_RAW_ROOT.exists()],
    ['Pacotes DCASE', ', '.join(DCASE_PACKAGES), ''],
    ['MIMII root', str(MIMII_ROOT), MIMII_ROOT.exists()],
    ['MIMII download automático', AUTO_DOWNLOAD_MIMII, 'desligado por padrão porque cada zip oficial tem cerca de 6 a 10 GB'],
    ['MIMII pacotes selecionados', ', '.join(MIMII_PACKAGES), f'recomendado começar por {MIMII_RECOMMENDED_PACKAGE}'],
    ['MIMII modo amostral remoto', MIMII_REMOTE_SAMPLE_ENABLED, f'{MIMII_REMOTE_SAMPLE_SIZE} áudios por pacote quando o download automático estiver ligado'],
    ['MIMII amostra por classe', MAX_MIMII_PER_LABEL, 'cap por classe após a descoberta local/remota'],
    ['MIMII total máximo descoberto', MAX_MIMII_TOTAL_FILES, 'limite total do catálogo MIMII dentro do notebook'],
    ['Kaggle cego por classe', MAX_KAGGLE_BLIND_PER_CLASS, INCLUDE_KAGGLE_BLIND],
    ['DCASE cego por máquina/classe', MAX_DCASE_BLIND_PER_MACHINE_LABEL, INCLUDE_DCASE_BLIND],
    ['MIMII cego por classe', MAX_MIMII_BLIND_PER_LABEL, INCLUDE_MIMII_BLIND],
    ['Mixup supervisionado', MIXUP_ENABLED, f'ratio={MIXUP_RATIO} | lambda={MIXUP_LAMBDA} | fixo={MIXUP_USE_FIXED_LAMBDA}'],
    ['Mixup normal-normal (domínio)', DOMAIN_MIXUP_ENABLED, f'ratio={DOMAIN_MIXUP_RATIO} | peso={DOMAIN_MIXUP_SAMPLE_WEIGHT} | pool_unsup={DOMAIN_MIXUP_USE_UNSUP_POOL}'],
    ['OE auxiliar', OE_ENABLED, f'max_janelas={OE_MAX_WINDOWS} | peso={OE_SAMPLE_WEIGHT} | desligado por padrão quando False'],
    ['Busca leve de hiperparâmetros', ENABLE_HYPERPARAM_SEARCH, HYPERPARAM_RANKING_METRIC],
    ['GMM investigativo', ENABLE_GMM_CHALLENGER, f'max_componentes={GMM_MAX_COMPONENTS} | cov={GMM_COVARIANCE_TYPE}'],
    ['Isolation Forest investigativo', ENABLE_IFOREST_CHALLENGER, f'n_estimators={IFOREST_N_ESTIMATORS} | max_samples={IFOREST_MAX_SAMPLES}'],
    ['Tiny-AST auxiliar', ENABLE_TINY_AST_AUX, f'embed={TINY_AST_EMBED_DIM} | epocas={TINY_AST_EPOCHS} | max_janelas={TINY_AST_MAX_TRAIN_WINDOWS}'],
    ['Janela', f'{WINDOW_SEC}s / hop {HOP_SEC}s', ''],
])
config_df.columns = ['Item', 'Valor', 'Existe?']
display(config_df)


In [ ]:
# ============================================================
# 4. Download opcional das fontes públicas oficiais
# ============================================================
# DCASE baixa por padrão os pacotes selecionados.
# Para o MIMII, o notebook agora prioriza uma amostragem remota de arquivos
# quando o download automático estiver ativado, evitando puxar 6 a 10 GB inteiros
# sem necessidade. Se preferir o pacote completo, basta desligar o modo amostral remoto.

ZENODO_API = 'https://zenodo.org/api/records/{record_id}'
REMOTE_AUDIO_EXTS = {'.wav', '.flac', '.ogg', '.mp3'}


def zenodo_files(record_id: str) -> pd.DataFrame:
    response = requests.get(ZENODO_API.format(record_id=record_id), timeout=60)
    response.raise_for_status()
    data = response.json()
    rows = []
    for f in data.get('files', []):
        links = f.get('links', {})
        rows.append({
            'record_id': record_id,
            'key': f.get('key'),
            'size_mb': round(float(f.get('size', 0)) / 1024 / 1024, 2),
            'download_url': links.get('self') or links.get('content'),
            'checksum': f.get('checksum'),
        })
    return pd.DataFrame(rows)


def download_file(url: str, out_path: Path, chunk_size: int = 1024 * 1024) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        print('Já existe, pulando:', out_path.name)
        return out_path
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(out_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=out_path.name) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return out_path


def extract_zip(zip_path: Path, extract_root: Path) -> None:
    marker = extract_root / f'.extracted_{zip_path.stem}'
    if marker.exists():
        print('Já extraído:', zip_path.name)
        return
    print('Extraindo:', zip_path.name)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_root)
    marker.write_text('ok')


def extract_remote_zip_sample(url: str, sample_root: Path, max_files: int, seed: int = RANDOM_STATE) -> pd.DataFrame:
    try:
        from remotezip import RemoteZip
    except Exception as exc:
        print('A biblioteca `remotezip` não está disponível para amostragem remota do MIMII:', exc)
        return pd.DataFrame()

    import shutil

    sample_root = Path(sample_root)
    manifest_path = sample_root / '_manifest.csv'
    if manifest_path.exists():
        try:
            manifest = pd.read_csv(manifest_path)
            if not manifest.empty:
                print(f'Amostra remota já preparada em {sample_root}: {len(manifest)} arquivos.')
                return manifest
        except Exception:
            pass

    sample_root.mkdir(parents=True, exist_ok=True)
    try:
        with RemoteZip(url, initial_buffer_size=256 * 1024) as zf:
            names = [
                name for name in zf.namelist()
                if (not name.endswith('/')) and (Path(name).suffix.lower() in REMOTE_AUDIO_EXTS)
            ]
            if not names:
                print('Nenhum áudio foi encontrado dentro do zip remoto do MIMII.')
                return pd.DataFrame()

            rng_local = np.random.default_rng(seed)
            sample_size = min(max_files, len(names))
            picked = sorted(rng_local.choice(np.asarray(names, dtype=object), size=sample_size, replace=False).tolist())

            rows = []
            for member in tqdm(picked, desc=f'Amostra remota {sample_root.name}'):
                out_path = sample_root / member
                out_path.parent.mkdir(parents=True, exist_ok=True)
                if not out_path.exists():
                    with zf.open(member) as src, open(out_path, 'wb') as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                rows.append({
                    'member': member,
                    'local_path': str(out_path),
                    'size_bytes': out_path.stat().st_size if out_path.exists() else 0,
                })
    except Exception as exc:
        print('Falha ao montar a amostra remota do MIMII:', exc)
        print('Se preferir, desligue o modo remoto e use o download completo do zip.')
        return pd.DataFrame()

    manifest = pd.DataFrame(rows)
    if not manifest.empty:
        manifest.to_csv(manifest_path, index=False)
    return manifest


try:
    dcase_file_table = zenodo_files(DCASE_RECORD_ID)
    display(dcase_file_table[['key', 'size_mb']])
except Exception as exc:
    dcase_file_table = pd.DataFrame()
    print('Não foi possível consultar o Zenodo:', exc)

if AUTO_DOWNLOAD_DCASE and not dcase_file_table.empty:
    selected = dcase_file_table[dcase_file_table['key'].isin(DCASE_PACKAGES)]
    for _, row in selected.iterrows():
        zip_path = DCASE_DOWNLOAD_ROOT / row['key']
        download_file(row['download_url'], zip_path)
        extract_zip(zip_path, DCASE_EXTRACT_ROOT)
else:
    print('Download automático do DCASE desativado ou lista indisponível.')

try:
    mimii_table = zenodo_files(MIMII_RECORD_ID)
    mimii_table = mimii_table.sort_values('size_mb').reset_index(drop=True)
    display(Markdown('### Catálogo oficial do MIMII no Zenodo'))
    display(mimii_table[['key', 'size_mb']])
except Exception as exc:
    mimii_table = pd.DataFrame()
    print('Não foi possível consultar o catálogo do MIMII no Zenodo:', exc)

if AUTO_DOWNLOAD_MIMII:
    if mimii_table.empty:
        print('MIMII: catálogo indisponível, então o download automático não pôde começar.')
    else:
        available_keys = set(mimii_table['key'])
        missing = [pkg for pkg in MIMII_PACKAGES if pkg not in available_keys]
        if missing:
            print('Pacotes MIMII não encontrados no Zenodo:', missing)
            print('Use exatamente um dos nomes exibidos na tabela acima.')

        selected = mimii_table[mimii_table['key'].isin(MIMII_PACKAGES)].copy()
        total_gb = selected['size_mb'].sum() / 1024.0 if not selected.empty else 0.0
        if selected.empty:
            print('MIMII: nenhum pacote foi selecionado para download. Ajuste AAD_MIMII_PACKAGES.')
        else:
            display(Markdown(
                f'**Download MIMII ativado** | pacotes: `{", ".join(selected["key"].tolist())}` | tamanho total aproximado: `{total_gb:.2f} GB`'
            ))

            if MIMII_REMOTE_SAMPLE_ENABLED:
                display(Markdown(
                    f'**Modo amostral remoto ligado** | o notebook vai extrair apenas cerca de `{MIMII_REMOTE_SAMPLE_SIZE}` áudios por pacote, em vez de baixar o zip inteiro.'
                ))
                sample_manifests = []
                for package_idx, (_, row) in enumerate(selected.iterrows(), start=1):
                    sample_root = MIMII_ROOT / 'remote_samples' / Path(row['key']).stem
                    manifest = extract_remote_zip_sample(
                        row['download_url'],
                        sample_root,
                        max_files=MIMII_REMOTE_SAMPLE_SIZE,
                        seed=RANDOM_STATE + package_idx,
                    )
                    if not manifest.empty:
                        manifest['package'] = row['key']
                        sample_manifests.append(manifest)

                if sample_manifests:
                    sample_manifest_df = pd.concat(sample_manifests, ignore_index=True)
                    display(Markdown('### Resumo da amostra remota do MIMII'))
                    display(sample_manifest_df.groupby('package').size().reset_index(name='áudios_extraídos'))
                else:
                    print('A amostragem remota do MIMII não gerou arquivos. Veja as mensagens acima.')
            else:
                display(Markdown('**Modo amostral remoto desligado** | o notebook fará o download completo dos zips selecionados.'))
                for _, row in selected.iterrows():
                    zip_path = MIMII_ROOT / 'zips' / row['key']
                    download_file(row['download_url'], zip_path)
                    extract_zip(zip_path, MIMII_ROOT)
else:
    print('MIMII: download automático desativado por padrão para evitar baixar 6 a 10 GB sem querer.')
    print('Para ativar no Colab e já usar a amostra remota, defina: AAD_AUTO_DOWNLOAD_MIMII=1')
    print(f'Pacote recomendado para começar: {MIMII_RECOMMENDED_PACKAGE}')


In [ ]:
# ============================================================
# 5. Catálogo único de arquivos: Drive próprio + Kaggle + DCASE + MIMII
# ============================================================
# Cada linha do catálogo é um arquivo de áudio/sinal.
# Rótulos são inferidos por nome de pasta/arquivo: normal, anomaly, anomalo, fault, falha etc.
# Arquivos sem rótulo podem entrar no teste cego, mas não entram no cálculo de métrica.

AUDIO_EXTS = {'.wav', '.flac', '.ogg', '.mp3'}
SIGNAL_EXTS = AUDIO_EXTS | {'.csv', '.npy'}


def sample_df(df: pd.DataFrame, n: int, group_cols: list[str], seed: int = RANDOM_STATE) -> pd.DataFrame:
    if df.empty or n <= 0:
        return df
    parts = []
    for _, part in df.groupby(group_cols, dropna=False):
        parts.append(part.sample(n=min(n, len(part)), random_state=seed))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0]


def label_from_text(text: str) -> Optional[int]:
    text = text.lower()
    anomaly_tokens = ['anomaly', 'anomalous', 'abnormal', 'fault', 'broken', 'damage', 'falha', 'anomalia', 'anormal', 'defeito']
    normal_tokens = ['normal', 'healthy', 'ok', 'saudavel', 'saudável', 'bom']
    if any(tok in text for tok in anomaly_tokens):
        return 1
    if any(tok in text for tok in normal_tokens):
        return 0
    return None


def make_row(path: Path, dataset: str, source: str, machine: str, split_hint: str, label: Optional[int], group_prefix: str) -> dict:
    label_value = -1 if label is None else int(label)
    return {
        'path': str(path),
        'dataset': dataset,
        'source': source,
        'machine': machine,
        'split_hint': split_hint,
        'label': label_value,
        'label_name': 'Sem rótulo' if label is None else ('Anômalo' if int(label) == 1 else 'Normal'),
        'group_id': f'{group_prefix}/{path.stem}',
        'file_name': path.name,
    }


def discover_drive_audio(dirs: list[Path], is_blind: bool = False) -> pd.DataFrame:
    rows = []
    for root in dirs:
        if not root.exists():
            continue
        files = [p for p in sorted(root.rglob('*')) if p.is_file() and p.suffix.lower() in SIGNAL_EXTS]
        if is_blind and len(files) > MAX_BLIND_FILES:
            files = list(pd.Series(files).sample(MAX_BLIND_FILES, random_state=RANDOM_STATE))
        for p in files:
            rel_text = str(p.relative_to(root)) if root in p.parents else str(p)
            label = label_from_text(rel_text)
            machine = p.parent.name if p.parent.name else 'maquina_propria'
            rows.append(make_row(
                p,
                dataset='Teste cego próprio' if is_blind else 'Áudios próprios do Drive',
                source='Drive cego' if is_blind else 'Drive próprio',
                machine=machine,
                split_hint='blind_domain_shift' if is_blind else ('own_labeled' if label is not None else 'own_unlabeled'),
                label=label,
                group_prefix=('blind_drive' if is_blind else 'own_drive') + '/' + root.name,
            ))
    df = pd.DataFrame(rows)
    if not is_blind and not df.empty:
        labeled = df[df['label'] >= 0]
        unlabeled = df[df['label'] < 0]
        labeled = sample_df(labeled, MAX_OWN_PER_LABEL, ['label']) if not labeled.empty else labeled
        df = pd.concat([labeled, unlabeled], ignore_index=True)
    return df


def discover_kaggle(root: Path) -> pd.DataFrame:
    rows = []
    if not kaggle_root_is_valid(root):
        print('Kaggle não encontrado no padrão esperado:', root)
        return pd.DataFrame(rows)
    folder_map = {'Normal': 0, 'Inner Race Fault': 1, 'Outer Race Fault': 1}
    for folder_name, label in folder_map.items():
        folder = root / folder_name
        for p in sorted(folder.glob('*.csv')):
            rows.append(make_row(p, 'Kaggle Bearing', 'Kaggle', 'bearing', 'kaggle_labeled', label, f'kaggle/{folder_name}'))
    df = pd.DataFrame(rows)
    return sample_df(df, MAX_KAGGLE_PER_CLASS, ['label']) if not df.empty else df


def discover_dcase(root_candidates: Iterable[Path]) -> pd.DataFrame:
    rows = []
    machine_dirs = []
    for root in root_candidates:
        if root.exists():
            machine_dirs.extend([p for p in root.rglob('*') if p.is_dir() and ((p / 'train').exists() or (p / 'test').exists())])
    seen = set()
    for machine_dir in sorted(machine_dirs):
        if machine_dir.resolve() in seen:
            continue
        seen.add(machine_dir.resolve())
        machine = machine_dir.name.replace('dev_', '')
        train_dir = machine_dir / 'train'
        test_dir = machine_dir / 'test'
        if train_dir.exists():
            temp = []
            for p in sorted(train_dir.rglob('*')):
                if p.suffix.lower() in AUDIO_EXTS:
                    temp.append(make_row(p, 'DCASE 2025 Task 2', 'DCASE', machine, 'dcase_train_normal', 0, f'dcase/{machine}/train'))
            if temp:
                rows.extend(sample_df(pd.DataFrame(temp), MAX_DCASE_TRAIN_PER_MACHINE, ['machine']).to_dict('records'))
        if test_dir.exists():
            temp = []
            for p in sorted(test_dir.rglob('*')):
                if p.suffix.lower() not in AUDIO_EXTS:
                    continue
                label = label_from_text(p.name)
                if label is None:
                    continue
                temp.append(make_row(p, 'DCASE 2025 Task 2', 'DCASE', machine, 'dcase_test_external', label, f'dcase/{machine}/test'))
            if temp:
                rows.extend(sample_df(pd.DataFrame(temp), MAX_DCASE_TEST_PER_MACHINE_LABEL, ['machine', 'label']).to_dict('records'))
    return pd.DataFrame(rows)


def discover_mimii(root: Path) -> pd.DataFrame:
    rows = []
    if not ENABLE_MIMII_SCAN or not root.exists():
        print('MIMII não encontrado ou varredura desativada:', root)
        return pd.DataFrame(rows)
    for p in sorted(root.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in AUDIO_EXTS:
            continue
        rel_text = '/'.join(p.parts[-10:])
        label = label_from_text(rel_text)
        if label is None:
            continue
        machine = next((part for part in p.parts if part.lower() in {'fan', 'pump', 'slider', 'valve'}), 'machine')
        rows.append(make_row(p, 'MIMII pequena amostra', 'MIMII', machine, 'mimii_labeled_sample', label, f'mimii/{machine}'))
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if MAX_MIMII_TOTAL_FILES > 0:
        label_groups = max(1, df['label'].nunique())
        label_cap = max(1, min(MAX_MIMII_PER_LABEL, int(np.ceil(MAX_MIMII_TOTAL_FILES / label_groups))))
        df = sample_df(df, label_cap, ['label'])
        if len(df) > MAX_MIMII_TOTAL_FILES:
            final_cap = max(1, int(np.ceil(MAX_MIMII_TOTAL_FILES / label_groups)))
            df = sample_df(df, final_cap, ['label'])
    return df.reset_index(drop=True)

catalog_parts = [
    discover_drive_audio(OWN_DATA_DIRS, is_blind=False),
    discover_drive_audio(BLIND_TEST_DIRS, is_blind=True),
    discover_kaggle(KAGGLE_ROOT),
    discover_dcase([DCASE_EXTRACT_ROOT, DCASE_RAW_ROOT, DATASETS_ROOT / 'dcase2025_task2']),
    discover_mimii(MIMII_ROOT),
]

catalog = pd.concat([df for df in catalog_parts if df is not None and not df.empty], ignore_index=True) if any(df is not None and not df.empty for df in catalog_parts) else pd.DataFrame()
if catalog.empty:
    raise RuntimeError('Nenhum arquivo encontrado. Verifique Drive, Kaggle, DCASE ou MIMII.')

catalog['source_id'] = np.arange(len(catalog))
catalog['label'] = catalog['label'].astype(int)
print('Arquivos selecionados:', len(catalog))
display(catalog.groupby(['dataset', 'source', 'split_hint', 'label_name']).size().reset_index(name='arquivos'))
display(catalog.head(12))


In [ ]:
# ============================================================
# 6. Separação sem vazamento de dados
# ============================================================
# - Supervisionado: aprende com Drive próprio rotulado, Kaggle e MIMII amostrado.
# - Não supervisionado: aprende somente com normalidade.
# - Avaliação externa padrão: DCASE test rotulado não usado no treino.
# - Teste cego multi-fonte: Drive cego + subconjunto cego do Kaggle + subconjunto cego do DCASE + subconjunto cego do MIMII.

catalog = catalog.copy()
catalog['role_supervised'] = 'nao_usado'
catalog['role_unsup_train'] = False
catalog['role_external_eval'] = False
catalog['role_blind_test'] = catalog['split_hint'].eq('blind_domain_shift')
catalog['role_oe_train'] = False
catalog['blind_origin'] = np.where(catalog['role_blind_test'], 'Drive cego', 'Não se aplica')


def mark_blind_subset(
    df: pd.DataFrame,
    mask: pd.Series,
    n_per_group: int,
    group_cols: list[str],
    origin: str,
    seed: int,
) -> set[int]:
    candidates = df[mask & (~df['role_blind_test'])].copy()
    if candidates.empty or n_per_group <= 0:
        return set()
    picked = sample_df(candidates, n_per_group, group_cols, seed=seed)
    ids = set(picked['source_id'])
    if ids:
        df.loc[df['source_id'].isin(ids), 'role_blind_test'] = True
        df.loc[df['source_id'].isin(ids), 'blind_origin'] = origin
    return ids


kaggle_blind_ids = set()
if INCLUDE_KAGGLE_BLIND:
    kaggle_blind_ids = mark_blind_subset(
        catalog,
        catalog['source'].eq('Kaggle') & catalog['label'].isin([0, 1]),
        MAX_KAGGLE_BLIND_PER_CLASS,
        ['label'],
        'Kaggle cego',
        RANDOM_STATE + 17,
    )

dcase_blind_ids = set()
if INCLUDE_DCASE_BLIND:
    dcase_blind_ids = mark_blind_subset(
        catalog,
        catalog['source'].eq('DCASE') &
        catalog['split_hint'].eq('dcase_test_external') &
        catalog['label'].isin([0, 1]),
        MAX_DCASE_BLIND_PER_MACHINE_LABEL,
        ['machine', 'label'],
        'DCASE cego',
        RANDOM_STATE + 23,
    )

mimii_blind_ids = set()
if INCLUDE_MIMII_BLIND:
    mimii_blind_ids = mark_blind_subset(
        catalog,
        catalog['source'].eq('MIMII') & catalog['label'].isin([0, 1]),
        MAX_MIMII_BLIND_PER_LABEL,
        ['machine', 'label'],
        'MIMII cego',
        RANDOM_STATE + 31,
    )

trainable_sup = catalog[
    catalog['source'].isin(['Drive próprio', 'Kaggle', 'MIMII']) &
    catalog['label'].isin([0, 1]) &
    (~catalog['role_blind_test'])
].copy()

if trainable_sup['label'].nunique() >= 2 and len(trainable_sup) >= 4:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
    idx_train, idx_test = next(splitter.split(trainable_sup, trainable_sup['label'], groups=trainable_sup['group_id']))
    sup_train_ids = set(trainable_sup.iloc[idx_train]['source_id'])
    sup_test_ids = set(trainable_sup.iloc[idx_test]['source_id'])
    catalog.loc[catalog['source_id'].isin(sup_train_ids), 'role_supervised'] = 'train'
    catalog.loc[catalog['source_id'].isin(sup_test_ids), 'role_supervised'] = 'test_interno'
else:
    print('Dados supervisionados insuficientes para split interno. O XGBoost pode ser pulado.')

catalog.loc[
    ((catalog['role_supervised'] == 'train') & (catalog['label'] == 0)) |
    ((catalog['split_hint'] == 'dcase_train_normal') & (~catalog['role_blind_test'])),
    'role_unsup_train'
] = True

catalog.loc[
    (
        (catalog['split_hint'] == 'dcase_test_external') &
        (~catalog['role_blind_test'])
    ) & catalog['label'].isin([0, 1]),
    'role_external_eval'
] = True

if OE_ENABLED:
    oe_mask = (
        (~catalog['role_blind_test']) &
        (~catalog['role_external_eval']) &
        (catalog['role_supervised'] != 'test_interno') &
        (
            (catalog['split_hint'].isin(['own_unlabeled'])) |
            ((catalog['source'] == 'DCASE') & (catalog['split_hint'] == 'dcase_train_normal')) |
            ((catalog['source'] == 'MIMII') & (catalog['role_supervised'] == 'nao_usado'))
        )
    )
    catalog.loc[oe_mask, 'role_oe_train'] = True


def assert_no_group_overlap(df: pd.DataFrame, left_mask: pd.Series, right_mask: pd.Series, left_name: str, right_name: str):
    left_groups = set(df.loc[left_mask, 'group_id'])
    right_groups = set(df.loc[right_mask, 'group_id'])
    overlap = sorted(left_groups & right_groups)
    if overlap:
        preview = overlap[:10]
        raise RuntimeError(
            f'Vazamento detectado entre {left_name} e {right_name}. '
            f'Grupos em comum: {preview} | total={len(overlap)}'
        )
    print(f'Sem vazamento entre {left_name} e {right_name}: {len(left_groups)} vs {len(right_groups)} grupos.')


assert_no_group_overlap(catalog, catalog['role_supervised'].eq('train'), catalog['role_supervised'].eq('test_interno'), 'treino supervisionado', 'teste interno')
assert_no_group_overlap(catalog, catalog['role_supervised'].eq('train'), catalog['role_blind_test'], 'treino supervisionado', 'teste cego')
assert_no_group_overlap(catalog, catalog['role_supervised'].eq('test_interno'), catalog['role_blind_test'], 'teste interno', 'teste cego')
assert_no_group_overlap(catalog, catalog['role_unsup_train'], catalog['role_external_eval'], 'treino não supervisionado', 'avaliação externa')
assert_no_group_overlap(catalog, catalog['role_unsup_train'], catalog['role_blind_test'], 'treino não supervisionado', 'teste cego')
assert_no_group_overlap(catalog, catalog['role_external_eval'], catalog['role_blind_test'], 'avaliação externa', 'teste cego')
assert_no_group_overlap(catalog, catalog['role_oe_train'], catalog['role_blind_test'], 'OE auxiliar', 'teste cego')

role_table = catalog.groupby([
    'source', 'split_hint', 'label_name', 'role_supervised', 'role_unsup_train',
    'role_external_eval', 'role_blind_test', 'role_oe_train', 'blind_origin'
]).size().reset_index(name='arquivos')
display(role_table)

blind_catalog = catalog[catalog['role_blind_test']].copy()
if not blind_catalog.empty:
    display(Markdown('### Composição do teste cego multi-fonte'))
    blind_table = blind_catalog.groupby(['blind_origin', 'source', 'dataset', 'label_name']).size().reset_index(name='arquivos_cegos')
    display(blind_table)
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.barplot(data=blind_table, y='blind_origin', x='arquivos_cegos', hue='label_name', ax=ax)
    ax.set_title('De onde vêm os arquivos do teste cego?')
    ax.set_xlabel('Quantidade de arquivos')
    ax.set_ylabel('Origem cega')
    plt.tight_layout(); plt.show()
    display(Markdown(f'''
    **Observação metodológica:** o teste cego do `Drive` pode conter apenas ruído de fundo ou mudança de ambiente, sem falha real evidente. Por isso este notebook agora inclui também um `blind benchmark` com `Kaggle`, `DCASE` e `MIMII`, usando apenas arquivos nunca marcados para treino.

    - `Kaggle cego` reservado: `{len(kaggle_blind_ids)}` arquivos.
    - `DCASE cego` reservado: `{len(dcase_blind_ids)}` arquivos.
    - `MIMII cego` reservado: `{len(mimii_blind_ids)}` arquivos.
    - `Drive cego` continua útil como teste operacional de mudança de domínio, mesmo quando não houver rótulo ou anomalia clara.
    '''))

oe_catalog = catalog[catalog['role_oe_train']].copy()
if OE_ENABLED:
    display(Markdown('### Pool do OE auxiliar'))
    if oe_catalog.empty:
        display(Markdown('**OE ativado, mas sem arquivos elegíveis.** Isso não significa erro de código; significa apenas que, com a separação anti-vazamento atual, não sobrou material seguro para OE.'))
    else:
        display(oe_catalog.groupby(['source', 'dataset', 'label_name']).size().reset_index(name='arquivos_oe'))

display(Markdown('''
**Leitura da separação:**

- `role_supervised=train`: arquivos usados para treinar o XGBoost.
- `role_unsup_train=True`: arquivos normais usados para aprender a bolha de normalidade do Mahalanobis/GMM/IForest.
- `role_external_eval=True`: avaliação externa padrão, separada do teste cego.
- `role_blind_test=True`: cenário cego multi-fonte, agora com `Drive`, `Kaggle cego`, `DCASE cego` e `MIMII cego`, sempre sem reutilizar arquivos marcados para treino.
- `role_oe_train=True`: dados auxiliares opcionais para Outlier Exposure. Eles não entram como teste; servem apenas para endurecer o supervisionado.
'''))


In [ ]:
# ============================================================
# 7. Visualização didática do conjunto de dados
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
count_source = catalog.groupby(['source', 'label_name']).size().reset_index(name='arquivos')
sns.barplot(data=count_source, x='source', y='arquivos', hue='label_name', ax=axes[0])
axes[0].set_title('Arquivos por fonte')
axes[0].set_xlabel('Fonte')
axes[0].set_ylabel('Número de arquivos')
axes[0].legend(title='Classe')

count_machine = catalog.groupby(['machine', 'label_name']).size().reset_index(name='arquivos')
sns.barplot(data=count_machine, x='machine', y='arquivos', hue='label_name', ax=axes[1])
axes[1].set_title('Arquivos por máquina')
axes[1].set_xlabel('Máquina')
axes[1].set_ylabel('Número de arquivos')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(title='Classe')
plt.tight_layout()
plt.show()

display(Markdown("""
**Como ler:** se uma classe tem muito mais arquivos que a outra, o modelo pode ficar enviesado. O Kaggle fornece falhas rotuladas, enquanto o DCASE fortalece a validação externa e o treino não supervisionado com normalidade.
"""))

In [ ]:
# ============================================================
# 7.1. Diagrama visual do pipeline para apresentação
# ============================================================
# Este desenho ajuda quem nunca viu DSP/ML a entender o caminho do áudio até a decisão.

steps = [
    ('Áudio bruto', 'som gravado\npor sensor/celular'),
    ('Janelas', f'pedaços de {WINDOW_SEC:.1f}s\ncom hop {HOP_SEC:.1f}s'),
    ('HHT + UKF', 'limpeza e\ntransientes'),
    ('RPCA', 'separa padrão\nestável e impactos'),
    ('Features', 'MFCC, Mel, RMS,\ncurtose, NMF'),
    ('Modelos', 'XGBoost\nMahalanobis+Gamma'),
    ('Decisão + XAI', 'alerta, métricas\ne explicação visual'),
]
fig, ax = plt.subplots(figsize=(18, 4.8))
ax.axis('off')
for i, (title, subtitle) in enumerate(steps):
    x = i * 1.65
    color = '#E8F3FF' if i < 5 else '#FFF1D8'
    rect = plt.Rectangle((x, 0.25), 1.35, 0.70, facecolor=color, edgecolor='#335C81', linewidth=1.8)
    ax.add_patch(rect)
    ax.text(x + 0.675, 0.72, title, ha='center', va='center', fontsize=13, fontweight='bold')
    ax.text(x + 0.675, 0.45, subtitle, ha='center', va='center', fontsize=10)
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 1.55, 0.60), xytext=(x + 1.35, 0.60), arrowprops=dict(arrowstyle='->', lw=2, color='#335C81'))
ax.set_xlim(-0.1, len(steps) * 1.65 - 0.15)
ax.set_ylim(0, 1.25)
ax.set_title('Visão geral: como o áudio vira um diagnóstico de anomalia', fontsize=18, fontweight='bold')
plt.tight_layout(); plt.show()
display(Markdown('''
**Leitura simples:** o notebook não tenta ouvir como uma pessoa. Ele corta o áudio em pequenos trechos, limpa ruídos, transforma cada trecho em números e compara esses números com padrões aprendidos de normalidade e falha. O XAI fecha o ciclo mostrando qual parte do som motivou o alerta.
'''))


In [ ]:
# ============================================================
# 8. Leitura de sinais, DSP campeão HHT + UKF e janelas
# ============================================================


def read_signal(path: str, source: str, target_sr: int = SR_TARGET) -> tuple[np.ndarray, int]:
    p = Path(path)
    suffix = p.suffix.lower()
    if suffix in AUDIO_EXTS:
        y, sr = librosa.load(p, sr=target_sr, mono=True)
        return y.astype(np.float32), target_sr
    if suffix == '.csv':
        df = pd.read_csv(p)
        numeric = df.select_dtypes(include=[np.number])
        if numeric.empty:
            raise ValueError(f'CSV sem coluna numérica: {p.name}')
        y = numeric.iloc[:, 0].to_numpy(dtype=np.float32)
        if CSV_SIGNAL_SR != target_sr:
            y = librosa.resample(y, orig_sr=CSV_SIGNAL_SR, target_sr=target_sr)
        return y.astype(np.float32), target_sr
    if suffix == '.npy':
        return np.load(p).astype(np.float32).ravel(), target_sr
    raise ValueError(f'Formato não suportado: {p.suffix}')


def normalize_signal(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    y = np.nan_to_num(y)
    y = y - np.mean(y)
    peak = np.max(np.abs(y)) + 1e-8
    return (y / peak).astype(np.float32)


def apply_ukf_denoising(y: np.ndarray, max_points: int = 4000) -> np.ndarray:
    y = normalize_signal(y)
    if not HAS_FILTERPY or len(y) < 8:
        win = min(len(y) // 2 * 2 - 1, 101)
        return scipy_signal.savgol_filter(y, window_length=win, polyorder=2).astype(np.float32) if win >= 7 else y
    idx = np.arange(len(y))
    if len(y) > max_points:
        control_idx = np.linspace(0, len(y) - 1, max_points).astype(int)
        y_control = y[control_idx]
    else:
        control_idx = idx
        y_control = y
    def fx(x, dt):
        return x
    def hx(x):
        return x
    points = MerweScaledSigmaPoints(n=1, alpha=0.1, beta=2.0, kappa=0.0)
    ukf = UKF(dim_x=1, dim_z=1, dt=1.0, fx=fx, hx=hx, points=points)
    ukf.x = np.array([float(y_control[0])])
    ukf.P *= 0.1
    ukf.Q *= 1e-5
    ukf.R *= 1e-2
    filtered = np.empty_like(y_control, dtype=np.float32)
    for i, z in enumerate(y_control):
        ukf.predict(); ukf.update(np.array([float(z)])); filtered[i] = ukf.x[0]
    if len(y) > max_points:
        filtered = np.interp(idx, control_idx, filtered).astype(np.float32)
    return normalize_signal(filtered)


def apply_hht_transient_isolation(y: np.ndarray, sr: int, use_true_emd: bool = False, max_emd_points: int = 12000) -> np.ndarray:
    y = normalize_signal(y)
    if use_true_emd and HAS_PYEMD and len(y) <= max_emd_points:
        try:
            imfs = EMD().emd(y)
            if imfs.ndim == 2 and imfs.shape[0] > 0:
                candidate = np.sum(imfs[:min(3, imfs.shape[0])], axis=0)
                envelope = np.abs(scipy_signal.hilbert(candidate))
                envelope = envelope / (np.percentile(envelope, 95) + 1e-8)
                return normalize_signal(candidate * np.clip(envelope, 0, 2))
        except Exception as exc:
            print('HHT/EMD real falhou; usando modo rápido:', exc)
    cutoff = max(30, min(120, sr // 100))
    b, a = scipy_signal.butter(4, cutoff / (sr / 2), btype='highpass')
    high = scipy_signal.filtfilt(b, a, y).astype(np.float32)
    envelope = np.abs(scipy_signal.hilbert(high))
    envelope = envelope / (np.percentile(envelope, 95) + 1e-8)
    return normalize_signal(high * np.clip(envelope, 0, 2))


def champion_dsp(y: np.ndarray, sr: int) -> np.ndarray:
    return normalize_signal(apply_ukf_denoising(apply_hht_transient_isolation(y, sr, use_true_emd=False)))


def sliding_windows(y: np.ndarray, sr: int, window_sec: float = WINDOW_SEC, hop_sec: float = HOP_SEC, max_windows: int = MAX_WINDOWS_PER_FILE) -> list[np.ndarray]:
    win = int(window_sec * sr)
    hop = int(hop_sec * sr)
    if len(y) < win:
        y = np.pad(y, (0, win - len(y)))
    starts = list(range(0, max(1, len(y) - win + 1), hop))
    if len(starts) > max_windows:
        starts = list(np.linspace(starts[0], starts[-1], max_windows).astype(int))
    return [y[s:s + win].astype(np.float32) for s in starts]

example_row = catalog.sample(1, random_state=RANDOM_STATE).iloc[0]
y_raw, sr = read_signal(example_row['path'], example_row['source'])
y_raw = normalize_signal(y_raw)
y_dsp = champion_dsp(y_raw, sr)
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
t = np.arange(min(len(y_raw), sr * 3)) / sr
axes[0].plot(t, y_raw[:len(t)], color='gray')
axes[0].set_title('Antes do DSP: sinal bruto')
axes[0].set_ylabel('Amplitude')
axes[1].plot(t, y_dsp[:len(t)], color='tab:blue')
axes[1].set_title('Depois do DSP campeão: HHT + UKF')
axes[1].set_xlabel('Tempo (s)')
axes[1].set_ylabel('Amplitude')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 8.1. Comparativo didático do DSP
# ============================================================
# Esta célula mostra visualmente o papel de cada tratamento do sinal.

example_row_dsp = catalog.sample(1, random_state=RANDOM_STATE + 7).iloc[0]
display(Markdown('### Áudio de exemplo usado no comparativo DSP'))
display(pd.DataFrame([{
    'Arquivo': example_row_dsp['file_name'],
    'Fonte': example_row_dsp['source'],
    'Base': example_row_dsp['dataset'],
    'Máquina': example_row_dsp['machine'],
    'Classe': example_row_dsp['label_name'],
}]))

y_raw_cmp, sr_cmp = read_signal(example_row_dsp['path'], example_row_dsp['source'])
y_raw_cmp = normalize_signal(y_raw_cmp)
y_hht_cmp = apply_hht_transient_isolation(y_raw_cmp, sr_cmp, use_true_emd=False)
y_ukf_cmp = apply_ukf_denoising(y_raw_cmp)
y_hht_ukf_cmp = champion_dsp(y_raw_cmp, sr_cmp)


def focus_interval(signal: np.ndarray, sr: int, window_ms: float = 120.0):
    idx = int(np.argmax(np.abs(signal)))
    center = idx / sr
    half = window_ms / 2000.0
    return max(0.0, center - half), min(len(signal) / sr, center + half), center


focus_start, focus_end, focus_center = focus_interval(y_hht_ukf_cmp, sr_cmp)
plot_seconds = min(len(y_raw_cmp) / sr_cmp, max(3.0, focus_end + 0.15))
plot_len = int(plot_seconds * sr_cmp)
t = np.arange(plot_len) / sr_cmp

signals_cmp = [
    ('Áudio bruto', y_raw_cmp[:plot_len], '#7A7A7A', 'Som original com ruído de fundo e transientes misturados.'),
    ('Após HHT', y_hht_cmp[:plot_len], '#D95F02', 'A HHT puxa eventos rápidos e não lineares para cima.'),
    ('Após UKF', y_ukf_cmp[:plot_len], '#1B9E77', 'O UKF reduz oscilação aleatória e suaviza o sensor.'),
    ('HHT + UKF', y_hht_ukf_cmp[:plot_len], '#3366CC', 'A combinação evidencia o choque e derruba o piso de ruído.'),
]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.suptitle(
    f'Comparativo DSP | {example_row_dsp["file_name"]} | {example_row_dsp["source"]} | {example_row_dsp["dataset"]}',
    fontsize=16,
    fontweight='bold',
    y=0.995,
)
for ax, (title, sig, color, note) in zip(axes, signals_cmp):
    ax.plot(t, sig, color=color, linewidth=1.2)
    ax.axvspan(focus_start, focus_end, color='gold', alpha=0.15)
    ax.axvline(focus_center, color='goldenrod', linestyle='--', linewidth=1.4)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Amplitude')
    ax.grid(True, linestyle='--', alpha=0.25)
    ax.text(
        0.99, 0.88, note, transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.25', fc='white', ec=color, alpha=0.9)
    )
axes[-1].set_xlabel('Tempo (s)')
plt.tight_layout(); plt.show()

display(Markdown('''
**Como ler este comparativo:** a faixa amarela marca a região mais informativa do áudio. No bruto, o evento raro compete com o ruído ambiente. A HHT destaca transientes curtos; o UKF abaixa a flutuação estocástica; e a fusão `HHT + UKF` aumenta o contraste entre falha e fundo. Esse é o motivo de usarmos DSP antes do modelo.
'''))


In [ ]:
# ============================================================
# 9. Extração do Super-Vector com RPCA
# ============================================================
# Super-Vector = MFCC + Mel + estatísticas físicas + RPCA + erro NMF.
# RPCA separa o espectrograma em:
# - baixa-rank: comportamento acústico repetitivo/estável da máquina;
# - esparso: impactos, transientes e eventos raros que costumam carregar a anomalia.
# Também preparamos uma imagem log-Mel compacta para o Tiny-AST auxiliar.

N_MELS = 48
N_MFCC = 13
NMF_COMPONENTS = 8
ENABLE_RPCA = os.getenv('AAD_ENABLE_RPCA', '1').strip().lower() in {'1', 'true', 'yes', 'sim'}
RPCA_MAX_ITER = int(os.getenv('AAD_RPCA_MAX_ITER', '8'))
RPCA_TOL = float(os.getenv('AAD_RPCA_TOL', '1e-4'))


def safe_stat(func, values: np.ndarray, default: float = 0.0) -> float:
    try:
        val = func(values)
        return float(val) if np.isfinite(val) else default
    except Exception:
        return default


def soft_threshold(x: np.ndarray, tau: float) -> np.ndarray:
    return np.sign(x) * np.maximum(np.abs(x) - tau, 0.0)


def robust_pca_ialm(matrix: np.ndarray, max_iter: int = RPCA_MAX_ITER, tol: float = RPCA_TOL) -> tuple[np.ndarray, np.ndarray]:
    M = np.nan_to_num(matrix.astype(np.float32), copy=False)
    if not ENABLE_RPCA or M.size == 0 or min(M.shape) < 2:
        return M, np.zeros_like(M, dtype=np.float32)

    lam = 1.0 / np.sqrt(max(M.shape))
    norm_two = float(np.linalg.norm(M, 2)) + 1e-8
    norm_inf = float(np.max(np.abs(M)) / lam) + 1e-8
    dual_norm = max(norm_two, norm_inf)
    Y = M / dual_norm
    L = np.zeros_like(M, dtype=np.float32)
    S = np.zeros_like(M, dtype=np.float32)
    mu = 1.25 / norm_two
    mu_bar = mu * 1e5
    rho = 1.5
    d_norm = float(np.linalg.norm(M, 'fro')) + 1e-8

    for _ in range(max_iter):
        U, sigma, VT = np.linalg.svd(M - S + (Y / mu), full_matrices=False)
        sigma_shrink = np.maximum(sigma - (1.0 / mu), 0.0)
        rank = int(np.sum(sigma_shrink > 0))
        if rank > 0:
            L = (U[:, :rank] * sigma_shrink[:rank]) @ VT[:rank, :]
        else:
            L = np.zeros_like(M, dtype=np.float32)
        S = soft_threshold(M - L + (Y / mu), lam / mu)
        residual = M - L - S
        Y = Y + mu * residual
        mu = min(mu * rho, mu_bar)
        if np.linalg.norm(residual, 'fro') / d_norm < tol:
            break
    return L.astype(np.float32), S.astype(np.float32)


def rpca_decompose_logmel(logmel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    centered = logmel.astype(np.float32) - float(np.median(logmel))
    scaled = centered / (float(np.std(centered)) + 1e-8)
    return robust_pca_ialm(scaled)


def add_rpca_features(feats: dict, logmel: np.ndarray, sr: int) -> None:
    low_rank, sparse = rpca_decompose_logmel(logmel)
    sparse_abs = np.abs(sparse)
    low_abs = np.abs(low_rank)
    total_energy = float(np.mean(np.abs(low_rank) + sparse_abs) + 1e-8)
    sparse_energy = float(np.mean(sparse_abs))
    low_energy = float(np.mean(low_abs))

    feats['rpca_sparse_energy'] = sparse_energy
    feats['rpca_low_rank_energy'] = low_energy
    feats['rpca_sparse_ratio'] = sparse_energy / total_energy
    feats['rpca_sparse_rms'] = float(np.sqrt(np.mean(sparse ** 2)))
    feats['rpca_sparse_p95'] = float(np.percentile(sparse_abs, 95))
    feats['rpca_sparse_max'] = float(np.max(sparse_abs))
    feats['rpca_sparse_kurtosis'] = safe_stat(stats.kurtosis, sparse.ravel())
    feats['rpca_sparse_skewness'] = safe_stat(stats.skew, sparse.ravel())
    feats['rpca_sparse_crest_factor'] = float((np.max(sparse_abs) + 1e-8) / (np.sqrt(np.mean(sparse ** 2)) + 1e-8))

    sparse_band = np.mean(sparse_abs, axis=1).astype(np.float32)
    mel_freqs = librosa.mel_frequencies(n_mels=logmel.shape[0], fmin=0, fmax=sr / 2)
    for low, high, name in [(0, 800, 'low'), (800, 2500, 'mid'), (2500, sr / 2, 'high')]:
        mask = (mel_freqs >= low) & (mel_freqs < high)
        feats[f'rpca_sparse_band_{name}'] = float(np.mean(sparse_band[mask])) if mask.any() else 0.0

    for i, val in enumerate(sparse_band):
        feats[f'rpca_sparse_mel_band_{i+1:02d}'] = float(val)


def build_ast_logmel_image(window: np.ndarray, sr: int, n_mels: int = AST_IMAGE_MELS, time_bins: int = AST_IMAGE_TIME_BINS) -> np.ndarray:
    y = normalize_signal(window)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=1024, hop_length=256, power=2.0)
    logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)
    if logmel.shape[1] != time_bins:
        logmel = scipy_signal.resample(logmel, time_bins, axis=1)
    logmel = np.nan_to_num(logmel.astype(np.float32), copy=False)
    p_low, p_high = np.percentile(logmel, [5, 95]) if logmel.size else (-80.0, 0.0)
    if not np.isfinite(p_low) or not np.isfinite(p_high) or abs(p_high - p_low) < 1e-8:
        p_low = float(np.min(logmel))
        p_high = float(np.max(logmel) + 1e-8)
    norm = np.clip((logmel - p_low) / (p_high - p_low + 1e-8), 0.0, 1.0)
    return norm.astype(np.float32)


def extract_base_features(window: np.ndarray, sr: int) -> tuple[dict, np.ndarray, np.ndarray]:
    y = normalize_signal(window)
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=1024, hop_length=256, power=2.0)
    mel_mean_power = np.mean(mel, axis=1).astype(np.float32)
    logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)
    mfcc = librosa.feature.mfcc(S=logmel, sr=sr, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    feats = {
        'rms_mean': safe_stat(np.mean, rms),
        'rms_std': safe_stat(np.std, rms),
        'zcr_mean': safe_stat(np.mean, zcr),
        'kurtosis': safe_stat(stats.kurtosis, y),
        'skewness': safe_stat(stats.skew, y),
        'peak_abs': float(np.max(np.abs(y))),
        'crest_factor': float((np.max(np.abs(y)) + 1e-8) / (np.sqrt(np.mean(y ** 2)) + 1e-8)),
        'spectral_centroid': safe_stat(np.mean, librosa.feature.spectral_centroid(y=y, sr=sr)[0]),
        'spectral_bandwidth': safe_stat(np.mean, librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]),
        'spectral_rolloff': safe_stat(np.mean, librosa.feature.spectral_rolloff(y=y, sr=sr)[0]),
        'spectral_flatness': safe_stat(np.mean, librosa.feature.spectral_flatness(y=y)[0]),
    }
    add_rpca_features(feats, logmel, sr)
    for i in range(N_MFCC):
        feats[f'mfcc_{i+1:02d}_mean'] = float(np.mean(mfcc[i]))
        feats[f'mfcc_{i+1:02d}_std'] = float(np.std(mfcc[i]))
        feats[f'delta_mfcc_{i+1:02d}_mean'] = float(np.mean(delta[i]))
        feats[f'delta_mfcc_{i+1:02d}_std'] = float(np.std(delta[i]))
    for i, val in enumerate(mel_mean_power):
        feats[f'mel_band_{i+1:02d}'] = float(val)
    ast_image = build_ast_logmel_image(y, sr)
    return feats, mel_mean_power, ast_image


def build_window_dataset(catalog: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    feature_rows, mel_rows, ast_rows, failures = [], [], [], []
    for _, row in tqdm(catalog.iterrows(), total=len(catalog), desc='Extraindo features'):
        try:
            y, sr = read_signal(row['path'], row['source'])
            y = champion_dsp(y, sr)
            for w_idx, win in enumerate(sliding_windows(y, sr)):
                feats, mel_vec, ast_image = extract_base_features(win, sr)
                feats.update({
                    'source_id': int(row['source_id']), 'window_id': int(w_idx), 'label': int(row['label']),
                    'label_name': row['label_name'], 'source': row['source'], 'dataset': row['dataset'],
                    'machine': row['machine'], 'group_id': row['group_id'], 'role_supervised': row['role_supervised'],
                    'role_unsup_train': bool(row['role_unsup_train']), 'role_external_eval': bool(row['role_external_eval']),
                    'role_blind_test': bool(row.get('role_blind_test', False)),
                    'blind_origin': row.get('blind_origin', 'Não se aplica'),
                    'file_name': row['file_name'], 'path': row['path'],
                })
                feature_rows.append(feats)
                mel_rows.append(mel_vec)
                ast_rows.append(ast_image)
        except Exception as exc:
            failures.append({'file': row.get('file_name'), 'error': str(exc)})
    if failures:
        print('Arquivos ignorados:', len(failures))
        display(pd.DataFrame(failures).head(10))
    if not feature_rows:
        raise RuntimeError('Nenhuma feature extraída.')
    return (
        pd.DataFrame(feature_rows),
        np.vstack(mel_rows).astype(np.float32),
        np.stack(ast_rows).astype(np.float32),
    )


window_df, mel_matrix, ast_matrix = build_window_dataset(catalog)
print('Janelas extraídas:', len(window_df))
print('RPCA ativo:', ENABLE_RPCA, '| iterações máximas:', RPCA_MAX_ITER)
print('Tensor auxiliar Tiny-AST:', ast_matrix.shape)
display(window_df.groupby(['source', 'label_name', 'role_supervised', 'role_unsup_train', 'role_external_eval']).size().reset_index(name='janelas'))

rpca_cols = [c for c in window_df.columns if c.startswith('rpca_')]
display(Markdown(f'### Descritores RPCA adicionados ao Super-Vector: {len(rpca_cols)}'))
display(pd.DataFrame({'feature_rpca': rpca_cols}).head(20))

channel_preview = pd.DataFrame([
    {'Canal': 'Canal A | Impulsivo', 'Quantidade de descritores': len([c for c in window_df.columns if c.startswith('rpca_') or c in {'rms_mean', 'rms_std', 'kurtosis', 'skewness', 'crest_factor', 'peak_abs'}])},
    {'Canal': 'Canal B | Espectral', 'Quantidade de descritores': len([c for c in window_df.columns if c.startswith('mfcc_') or c.startswith('delta_mfcc_')])},
    {'Canal': 'Canal C | Estrutural leve', 'Quantidade de descritores': len([c for c in window_df.columns if c.startswith('mel_band_') or c in {'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'zcr_mean'}])},
])
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(data=channel_preview, y='Canal', x='Quantidade de descritores', ax=axes[0], palette='viridis')
axes[0].set_title('Quantos descritores cada canal do Super-Vector entrega?')
axes[0].set_xlabel('Quantidade')
axes[0].set_ylabel('Canal')

sample_idx = 0 if len(ast_matrix) else None
if sample_idx is not None:
    axes[1].imshow(ast_matrix[sample_idx], aspect='auto', origin='lower', cmap='magma')
    axes[1].set_title('Imagem log-Mel auxiliar usada pelo Tiny-AST')
    axes[1].set_xlabel('Tempo compactado')
    axes[1].set_ylabel('Bandas Mel')
else:
    axes[1].text(0.5, 0.5, 'Sem imagem AST disponível', ha='center', va='center')
    axes[1].set_axis_off()
plt.tight_layout(); plt.show()

display(Markdown('''
**Leitura didática:** o `Super-Vector` continua sendo o coração leve do pipeline. A imagem `log-Mel` ao lado não substitui esse vetor; ela é um apoio visual e profundo para o `Tiny-AST` auxiliar, usado para enriquecer embeddings e construir um raio-X mais intuitivo do áudio.
'''))


In [ ]:
# ============================================================
# 9.1. RPCA visual e canais físicos do Super-Vetor
# ============================================================
# Esta célula traduz a engenharia de features para uma linguagem física.

example_row_feat = catalog.sample(1, random_state=RANDOM_STATE + 11).iloc[0]
display(Markdown('### Áudio de exemplo usado no RPCA e nos canais do Super-Vetor'))
display(pd.DataFrame([{
    'Arquivo': example_row_feat['file_name'],
    'Fonte': example_row_feat['source'],
    'Base': example_row_feat['dataset'],
    'Máquina': example_row_feat['machine'],
    'Classe': example_row_feat['label_name'],
}]))

y_feat, sr_feat = read_signal(example_row_feat['path'], example_row_feat['source'])
y_feat = champion_dsp(y_feat, sr_feat)
win_feat = sliding_windows(y_feat, sr_feat)[0]
mel_feat = librosa.feature.melspectrogram(y=normalize_signal(win_feat), sr=sr_feat, n_mels=N_MELS, n_fft=1024, hop_length=256, power=2.0)
logmel_feat = librosa.power_to_db(mel_feat + 1e-10, ref=np.max)
low_rank_feat, sparse_feat = rpca_decompose_logmel(logmel_feat)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    f'RPCA em uma janela de exemplo | {example_row_feat["file_name"]} | {example_row_feat["source"]}',
    fontsize=16,
    fontweight='bold',
    y=1.02,
)
for ax, mat, title, cmap in [
    (axes[0], logmel_feat, 'Espectrograma original', 'magma'),
    (axes[1], low_rank_feat, 'RPCA | Baixo posto (L)', 'viridis'),
    (axes[2], np.abs(sparse_feat), 'RPCA | Componente esparsa (S)', 'inferno'),
]:
    img = librosa.display.specshow(mat, x_axis='time', y_axis='mel', sr=sr_feat, hop_length=256, ax=ax, cmap=cmap)
    ax.set_title(title, fontsize=14, fontweight='bold')
    fig.colorbar(img, ax=ax, shrink=0.82)
plt.tight_layout(); plt.show()

preview_non_feature_cols = {
    'source_id', 'window_id', 'label', 'label_name', 'source', 'dataset', 'machine', 'group_id',
    'role_supervised', 'role_unsup_train', 'role_external_eval', 'role_blind_test', 'role_oe_train', 'file_name', 'path'
}
feature_columns_preview = globals().get('FEATURE_COLUMNS')
if not feature_columns_preview:
    feature_columns_preview = [
        c for c in window_df.columns
        if c not in preview_non_feature_cols and pd.api.types.is_numeric_dtype(window_df[c])
    ]
nmf_ready = 'nmf_reconstruction_error' in window_df.columns

channel_map = {
    'Canal A | Impulsivo (RPCA + estatísticas)': [c for c in feature_columns_preview if c.startswith('rpca_') or c in {'rms_mean', 'rms_std', 'kurtosis', 'skewness', 'crest_factor', 'peak_abs'}],
    'Canal B | Espectral (MFCC + deltas)': [c for c in feature_columns_preview if c.startswith('mfcc_') or c.startswith('delta_mfcc_')],
    'Canal C | Estrutural (NMF e forma espectral)': [c for c in feature_columns_preview if c.startswith('mel_band_') or c in {'nmf_reconstruction_error', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'zcr_mean'}],
}
channel_counts = pd.DataFrame({'Canal': list(channel_map.keys()), 'Nº de features': [len(v) for v in channel_map.values()]})

summary_rows = []
for channel_name, cols in channel_map.items():
    cols_valid = [c for c in cols if c in window_df.columns]
    if not cols_valid:
        continue
    mean_normal = float(window_df.loc[window_df['label'] == 0, cols_valid].mean().mean()) if (window_df['label'] == 0).any() else np.nan
    mean_anom = float(window_df.loc[window_df['label'] == 1, cols_valid].mean().mean()) if (window_df['label'] == 1).any() else np.nan
    summary_rows.append({'Canal': channel_name, 'Média normal': mean_normal, 'Média anômala': mean_anom, 'Diferença absoluta': abs(mean_anom - mean_normal) if np.isfinite(mean_anom) and np.isfinite(mean_normal) else np.nan})
channel_effect = pd.DataFrame(summary_rows)
if not nmf_ready:
    display(Markdown('**Aviso:** o `Canal C` ainda está parcial nesta visualização porque a célula `10. NMF` não foi executada. Rode a célula 10 e depois reexecute a 9.1 para ver o canal estrutural completo.'))

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
sns.barplot(data=channel_counts, x='Canal', y='Nº de features', ax=axes[0], palette='Blues_d')
axes[0].set_title('Composição do Super-Vetor por canal', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=18)
if not channel_effect.empty:
    plot_effect = channel_effect.melt(id_vars='Canal', value_vars=['Média normal', 'Média anômala'], var_name='Classe', value_name='Valor médio')
    sns.barplot(data=plot_effect, x='Canal', y='Valor médio', hue='Classe', ax=axes[1])
    axes[1].set_title('Leitura média dos canais: normal vs anômalo', fontsize=14, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=18)
plt.tight_layout(); plt.show()

display(pd.DataFrame([
    ['Canal A', 'Impulsivo', 'Mede agressividade do impacto: RMS, curtose e energia esparsa do RPCA.'],
    ['Canal B', 'Espectral', 'Lê o timbre da anomalia com MFCCs e deltas.'],
    ['Canal C', 'Estrutural', 'Mede quebra da normalidade com NMF e forma espectral.'],
], columns=['Canal', 'Natureza física', 'Explicação para leigos']))

display(Markdown('''
**Como explicar os 3 canais:**

- **Canal A** olha o quanto o choque é agressivo. Se a máquina bate, raspa ou pulsa, esse canal sobe.
- **Canal B** olha o timbre da falha. Ele ajuda a diferenciar atrito, ressonância e ruído estrutural.
- **Canal C** olha se o áudio quebrou a “receita” da máquina saudável. É onde o NMF ajuda bastante.

A combinação dos 3 canais evita depender de uma única pista acústica. Isso deixa a arquitetura mais robusta e mais explicável.
'''))


In [ ]:
# ============================================================
# 10. NMF: erro de reconstrução aprendido com sons normais
#     + Tiny-AST auxiliar para embeddings e saliência visual
# ============================================================

nmf_train_mask = (window_df['role_unsup_train']) & (window_df['label'] == 0)
if nmf_train_mask.sum() < max(5, NMF_COMPONENTS + 1):
    print('Poucas janelas normais para NMF. Usando erro zero.')
    window_df['nmf_reconstruction_error'] = 0.0
    nmf_model = None
    nmf_scaler = None
else:
    nmf_scaler = MinMaxScaler()
    mel_train = nmf_scaler.fit_transform(mel_matrix[nmf_train_mask.values])
    n_components = min(NMF_COMPONENTS, max(2, mel_train.shape[0] - 1), mel_train.shape[1])
    nmf_model = NMF(n_components=n_components, init='nndsvda', max_iter=500, random_state=RANDOM_STATE)
    nmf_model.fit(mel_train)
    mel_all = np.clip(nmf_scaler.transform(mel_matrix), 0, None)
    recon = nmf_model.inverse_transform(nmf_model.transform(mel_all))
    window_df['nmf_reconstruction_error'] = np.mean((mel_all - recon) ** 2, axis=1)


def parse_device_for_tiny_ast() -> Optional[torch.device]:
    if not HAS_TORCH:
        return None
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class TinyASTAutoencoder(nn.Module):
    def __init__(self, input_channels: int = 1, num_mels: int = AST_IMAGE_MELS, time_steps: int = AST_IMAGE_TIME_BINS,
                 embed_dim: int = TINY_AST_EMBED_DIM, num_heads: int = TINY_AST_HEADS, num_layers: int = TINY_AST_LAYERS):
        super().__init__()
        self.num_mels = num_mels
        self.time_steps = time_steps
        self.conv_encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.latent_h = num_mels // 4
        self.latent_w = time_steps // 4
        self.flat_dim = 64 * self.latent_h * self.latent_w
        self.project_to_embed = nn.Linear(self.flat_dim, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.project_from_embed = nn.Linear(embed_dim, self.flat_dim)
        self.conv_decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, input_channels, kernel_size=2, stride=2),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.size(0)
        x_enc = self.conv_encoder(x)
        x_flat = x_enc.view(batch_size, -1)
        x_embed = self.project_to_embed(x_flat).unsqueeze(1)
        return self.transformer_encoder(x_embed).squeeze(1)

    def forward(self, x: torch.Tensor, return_embedding: bool = False):
        embedding = self.encode(x)
        x_dec_flat = self.project_from_embed(embedding)
        x_dec = x_dec_flat.view(x.size(0), 64, self.latent_h, self.latent_w)
        recon = self.conv_decoder(x_dec)
        if return_embedding:
            return recon, embedding
        return recon


tiny_ast_model = None
tiny_ast_device = parse_device_for_tiny_ast()
tiny_ast_history = pd.DataFrame()
tiny_ast_feature_columns = []
tiny_ast_model_path = None


def compute_tiny_ast_features(ast_images: np.ndarray, batch_size: int = TINY_AST_BATCH_SIZE) -> tuple[np.ndarray, np.ndarray]:
    if tiny_ast_model is None or not HAS_TORCH or ast_images is None or len(ast_images) == 0:
        n = 0 if ast_images is None else len(ast_images)
        return np.zeros((n, 0), dtype=np.float32), np.zeros((n,), dtype=np.float32)
    model = tiny_ast_model
    model.eval()
    images = np.asarray(ast_images, dtype=np.float32)
    dataset = TensorDataset(torch.tensor(images[:, None, :, :], dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=max(1, batch_size), shuffle=False)
    embeddings, recon_errors = [], []
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(tiny_ast_device)
            recon, emb = model(batch, return_embedding=True)
            embeddings.append(emb.detach().cpu().numpy().astype(np.float32))
            recon_errors.append(((recon - batch) ** 2).mean(dim=(1, 2, 3)).detach().cpu().numpy().astype(np.float32))
    emb = np.vstack(embeddings) if embeddings else np.zeros((len(images), 0), dtype=np.float32)
    err = np.concatenate(recon_errors) if recon_errors else np.zeros((len(images),), dtype=np.float32)
    return emb, err


def reconstruct_tiny_ast_image(ast_image: np.ndarray) -> Optional[np.ndarray]:
    if tiny_ast_model is None or not HAS_TORCH:
        return None
    tiny_ast_model.eval()
    batch = torch.tensor(np.asarray(ast_image, dtype=np.float32)[None, None, :, :], dtype=torch.float32, device=tiny_ast_device)
    with torch.no_grad():
        recon, _ = tiny_ast_model(batch, return_embedding=True)
    return recon[0, 0].detach().cpu().numpy().astype(np.float32)


if ENABLE_TINY_AST_AUX and HAS_TORCH and len(ast_matrix) and nmf_train_mask.sum() >= 12:
    ast_train = ast_matrix[nmf_train_mask.values].astype(np.float32)
    if len(ast_train) > TINY_AST_MAX_TRAIN_WINDOWS:
        idx = rng.choice(len(ast_train), size=TINY_AST_MAX_TRAIN_WINDOWS, replace=False)
        ast_train = ast_train[idx]
    train_tensor = torch.tensor(ast_train[:, None, :, :], dtype=torch.float32)
    train_loader = DataLoader(TensorDataset(train_tensor), batch_size=max(1, TINY_AST_BATCH_SIZE), shuffle=True)

    tiny_ast_model = TinyASTAutoencoder().to(tiny_ast_device)
    optimizer = optim.Adam(tiny_ast_model.parameters(), lr=TINY_AST_LR, weight_decay=1e-5)
    criterion = nn.MSELoss()
    history_rows = []

    for epoch in range(1, max(1, TINY_AST_EPOCHS) + 1):
        tiny_ast_model.train()
        batch_losses = []
        for (batch,) in train_loader:
            batch = batch.to(tiny_ast_device)
            optimizer.zero_grad()
            recon = tiny_ast_model(batch)
            loss = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.item()))
        history_rows.append({'epoch': epoch, 'loss_medio': float(np.mean(batch_losses)) if batch_losses else np.nan})

    tiny_ast_history = pd.DataFrame(history_rows)
    ast_embeddings, ast_recon_error = compute_tiny_ast_features(ast_matrix)
    if ast_embeddings.size:
        for i in range(ast_embeddings.shape[1]):
            col = f'ast_emb_{i+1:02d}'
            window_df[col] = ast_embeddings[:, i]
            tiny_ast_feature_columns.append(col)
    window_df['tiny_ast_reconstruction_error'] = ast_recon_error
    tiny_ast_feature_columns.append('tiny_ast_reconstruction_error')

    display(Markdown(f'''### Tiny-AST auxiliar ativado

O `Tiny-AST` aqui **não é o classificador final**. Ele funciona como um reconstrutor leve treinado apenas com janelas normais, para fornecer duas peças extras: `embeddings profundos` e `erro de reconstrução`, que ajudam a explicar e enfrentar melhor o `domain shift`.'''))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.lineplot(data=tiny_ast_history, x='epoch', y='loss_medio', marker='o', ax=axes[0], color='tab:blue')
    axes[0].set_title('Treino do Tiny-AST auxiliar')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('MSE médio de reconstrução')

    sns.histplot(data=window_df, x='tiny_ast_reconstruction_error', hue='label_name', bins=40, kde=True, ax=axes[1])
    axes[1].set_title('Erro de reconstrução do Tiny-AST por classe')
    axes[1].set_xlabel('Erro de reconstrução')
    axes[1].set_ylabel('Janelas')
    plt.tight_layout(); plt.show()

    reference_image = ast_train[0]
    reference_recon = reconstruct_tiny_ast_image(reference_image)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(reference_image, aspect='auto', origin='lower', cmap='magma')
    axes[0].set_title('Imagem log-Mel de referência')
    axes[1].imshow(reference_recon, aspect='auto', origin='lower', cmap='magma')
    axes[1].set_title('Reconstrução feita pelo Tiny-AST')
    axes[2].imshow(np.abs(reference_image - reference_recon), aspect='auto', origin='lower', cmap='inferno')
    axes[2].set_title('Resíduo | o que o modelo não conseguiu explicar')
    for ax in axes:
        ax.set_xlabel('Tempo compactado')
        ax.set_ylabel('Bandas Mel')
    plt.tight_layout(); plt.show()
else:
    reason = 'desligado por parâmetro' if not ENABLE_TINY_AST_AUX else 'PyTorch indisponível' if not HAS_TORCH else 'poucas janelas normais para treino estável'
    display(Markdown(f'### Tiny-AST auxiliar não treinado: `{reason}`'))

non_feature_cols = {
    'source_id', 'window_id', 'label', 'label_name', 'source', 'dataset', 'machine', 'group_id',
    'role_supervised', 'role_unsup_train', 'role_external_eval', 'role_blind_test', 'role_oe_train', 'file_name', 'path'
}
FEATURE_COLUMNS = [c for c in window_df.columns if c not in non_feature_cols and pd.api.types.is_numeric_dtype(window_df[c])]
print('Features finais:', len(FEATURE_COLUMNS))
display(pd.DataFrame({'feature': FEATURE_COLUMNS}).head(25))

feature_family_df = pd.DataFrame([
    {'Família': 'Canal A | Impulsivo (RPCA)', 'Quantidade': len([c for c in FEATURE_COLUMNS if c.startswith('rpca_') or c in {'rms_mean', 'rms_std', 'kurtosis', 'skewness', 'crest_factor', 'peak_abs'}])},
    {'Família': 'Canal B | Espectral (MFCC + deltas)', 'Quantidade': len([c for c in FEATURE_COLUMNS if c.startswith('mfcc_') or c.startswith('delta_mfcc_')])},
    {'Família': 'Canal C | Estrutural (Mel + NMF)', 'Quantidade': len([c for c in FEATURE_COLUMNS if c.startswith('mel_band_') or c in {'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'zcr_mean', 'nmf_reconstruction_error'}])},
    {'Família': 'Canal D | Deep auxiliar (Tiny-AST)', 'Quantidade': len([c for c in FEATURE_COLUMNS if c.startswith('ast_emb_') or c == 'tiny_ast_reconstruction_error'])},
])
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(data=feature_family_df, y='Família', x='Quantidade', ax=axes[0], palette='crest')
axes[0].set_title('Quantos descritores entram na versão final do campeão?')
axes[0].set_xlabel('Quantidade')
axes[0].set_ylabel('Família de features')

sns.histplot(data=window_df, x='nmf_reconstruction_error', hue='label_name', bins=40, kde=True, ax=axes[1])
axes[1].set_title('Erro NMF: quanto o som foge da receita de normalidade?')
axes[1].set_xlabel('Erro NMF')
axes[1].set_ylabel('Janelas')
plt.tight_layout(); plt.show()

if 'tiny_ast_reconstruction_error' in window_df.columns:
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=window_df[['source', 'tiny_ast_reconstruction_error']].assign(source=lambda df: df['source'].astype(str)), x='source', y='tiny_ast_reconstruction_error', ax=ax)
    ax.set_title('Erro do Tiny-AST por fonte de áudio')
    ax.set_xlabel('Fonte')
    ax.set_ylabel('Erro de reconstrução')
    plt.xticks(rotation=15)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# 11. Métricas, pAUC e agregação por arquivo
# ============================================================

def safe_auc(y_true, scores, max_fpr=None):
    y_true = np.asarray(y_true).astype(int); scores = np.asarray(scores).astype(float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return float(roc_auc_score(y_true, scores, max_fpr=max_fpr)) if max_fpr else float(roc_auc_score(y_true, scores))
    except Exception:
        return np.nan


def metric_table(y_true, scores, preds, prefix='') -> dict:
    y_true = np.asarray(y_true).astype(int); preds = np.asarray(preds).astype(int); scores = np.asarray(scores).astype(float)
    return {
        'escopo': prefix, 'arquivos': int(len(y_true)),
        'accuracy': accuracy_score(y_true, preds) if len(y_true) else np.nan,
        'precision': precision_score(y_true, preds, zero_division=0) if len(y_true) else np.nan,
        'recall': recall_score(y_true, preds, zero_division=0) if len(y_true) else np.nan,
        'f1': f1_score(y_true, preds, zero_division=0) if len(y_true) else np.nan,
        'auc': safe_auc(y_true, scores), 'pauc_0_1': safe_auc(y_true, scores, max_fpr=0.10),
    }


def aggregate_file_scores(df: pd.DataFrame, score_col: str, threshold: Optional[float] = None) -> pd.DataFrame:
    agg = df.groupby('source_id').agg(
        score_mean=(score_col, 'mean'), score_p90=(score_col, lambda x: np.quantile(x, 0.90)),
        label=('label', 'first'), label_name=('label_name', 'first'), source=('source', 'first'),
        dataset=('dataset', 'first'), machine=('machine', 'first'), file_name=('file_name', 'first'),
        role_supervised=('role_supervised', 'first'), role_external_eval=('role_external_eval', 'first'),
        role_unsup_train=('role_unsup_train', 'first'),
        role_blind_test=('role_blind_test', 'first'),
        blind_origin=('blind_origin', 'first'),
    ).reset_index()
    if threshold is not None:
        agg['pred_threshold'] = (agg['score_p90'] >= threshold).astype(int)
    return agg


def plot_roc_pr(y_true, scores, title: str):
    y_true = np.asarray(y_true).astype(int); scores = np.asarray(scores).astype(float)
    if len(np.unique(y_true)) < 2:
        print('ROC/PR indisponível: apenas uma classe.'); return
    fpr, tpr, _ = roc_curve(y_true, scores)
    precision, recall, _ = precision_recall_curve(y_true, scores)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].plot(fpr, tpr, label=f'AUC={safe_auc(y_true, scores):.3f} | pAUC@0.1={safe_auc(y_true, scores, 0.1):.3f}')
    axes[0].axvspan(0, 0.10, color='green', alpha=0.12, label='Zona segura: FPR <= 10%')
    axes[0].plot([0, 1], [0, 1], '--', color='gray')
    axes[0].set_title(f'ROC - {title}'); axes[0].set_xlabel('Falso positivo'); axes[0].set_ylabel('Verdadeiro positivo'); axes[0].legend()
    axes[1].plot(recall, precision, label=f'AP={average_precision_score(y_true, scores):.3f}')
    axes[1].set_title(f'Precisão x Recall - {title}'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precisão'); axes[1].legend()
    plt.tight_layout(); plt.show()

display(Markdown('''
### Guia leigo das métricas

| Métrica | Pergunta que ela responde | Interpretação prática |
|---|---|---|
| Acurácia | De tudo que o modelo respondeu, quanto acertou? | Útil, mas pode enganar quando há poucas falhas. |
| Precisão | Quando ele grita anomalia, quantas vezes está certo? | Alta precisão significa menos alarmes falsos. |
| Recall | Das falhas reais, quantas ele conseguiu encontrar? | Alto recall significa menos falhas perdidas. |
| F1-Score | O equilíbrio entre precisão e recall está bom? | Bom resumo quando queremos detectar falha sem disparar alerta demais. |
| AUC-ROC | O score separa normal de anômalo em vários limiares? | Mede separação geral, antes de escolher o ponto de corte. |
| pAUC@0.1 | O modelo funciona na região de até 10% de falsos positivos? | É a métrica mais industrial: evita fadiga de alarmes no chão de fábrica. |
| Latência | Quanto tempo demora para responder? | Importante para Edge/TinyML e monitoramento em tempo quase real. |

**Regra prática:** no chão de fábrica, um modelo que acerta muitas falhas mas dispara falso alarme demais tende a ser desligado. Por isso a pAUC@0.1 e o limiar Gamma são tratados como peças centrais do pipeline.
'''))


In [ ]:
# ============================================================
# 12. Campeão supervisionado: XGBoost + L2 + Mixup + busca leve
# ============================================================
# Resultado 1: teste interno sem vazamento.
# Resultado 2: validação externa/domain shift quando houver dados rotulados.
# Frente 4: Mixup em espaço de features para endurecer a fronteira.
# Frente 7: OE opcional, separado da avaliação.
# Frente 11: busca leve de hiperparâmetros para reproduzir o refinamento final.

RESULT_REPORTS = []
SUPERVISED_FILE_REPORTS = {}
MIXUP_AUDIT = pd.DataFrame()
DOMAIN_MIXUP_AUDIT = pd.DataFrame()
OE_AUDIT = pd.DataFrame()
HYPERPARAM_SEARCH_RESULTS = pd.DataFrame()

sup_train = window_df[window_df['role_supervised'] == 'train'].copy()
sup_test = window_df[window_df['role_supervised'] == 'test_interno'].copy()
supervised_model = None
supervised_metrics = []


def make_mixup_rows(train_df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.Series, np.ndarray, pd.DataFrame]:
    # Cria pseudo-anomalias por interpolação entre normal e falha.
    if not MIXUP_ENABLED or train_df['label'].nunique() < 2:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()
    normal = train_df[train_df['label'] == 0]
    anomaly = train_df[train_df['label'] == 1]
    if normal.empty or anomaly.empty:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()
    n_mix = min(MIXUP_MAX_SAMPLES, max(1, int(len(train_df) * MIXUP_RATIO)))
    rng = np.random.default_rng(RANDOM_STATE)
    normal_idx = rng.choice(normal.index.to_numpy(), size=n_mix, replace=True)
    anomaly_idx = rng.choice(anomaly.index.to_numpy(), size=n_mix, replace=True)
    if MIXUP_USE_FIXED_LAMBDA:
        lambdas = np.full(n_mix, MIXUP_LAMBDA, dtype=np.float32)
    else:
        lambdas = rng.beta(MIXUP_ALPHA, MIXUP_ALPHA, size=n_mix).astype(np.float32)
    Xn = normal.loc[normal_idx, feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    Xa = anomaly.loc[anomaly_idx, feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    Xmix = (lambdas[:, None] * Xn) + ((1.0 - lambdas[:, None]) * Xa)
    mix_df = pd.DataFrame(Xmix, columns=feature_cols)
    y_mix = pd.Series(np.ones(n_mix, dtype=int), name='label')
    weights = np.full(n_mix, MIXUP_SAMPLE_WEIGHT, dtype=np.float32)
    audit = pd.DataFrame({'tecnica': ['Mixup'] * n_mix, 'lambda_normal': lambdas, 'peso_amostra': weights, 'rotulo_treino': ['Pseudo-anômalo'] * n_mix})
    return mix_df, y_mix, weights, audit


def make_domain_mixup_rows(train_df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.Series, np.ndarray, pd.DataFrame]:
    # Mistura exemplos normais de domínios diferentes para ampliar a noção de normalidade.
    if not DOMAIN_MIXUP_ENABLED:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()

    normal = train_df[train_df['label'] == 0].copy()
    if DOMAIN_MIXUP_USE_UNSUP_POOL and 'window_df' in globals():
        support = window_df[
            (window_df['label'] == 0) &
            (~window_df['role_blind_test']) &
            (~window_df['role_external_eval']) &
            (
                (window_df['role_supervised'] == 'train') |
                (window_df['role_unsup_train'])
            )
        ].copy()
        if not support.empty:
            normal = pd.concat([normal, support], ignore_index=True)
            dedup_cols = [c for c in ['source_id', 'window_id'] if c in normal.columns]
            if dedup_cols:
                normal = normal.drop_duplicates(subset=dedup_cols)

    if len(normal) < 2:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()

    normal['domain_key'] = normal[['source', 'dataset', 'machine', 'role_supervised']].fillna('NA').astype(str).agg(' | '.join, axis=1)
    normal['source_key'] = normal[['source', 'dataset']].fillna('NA').astype(str).agg(' | '.join, axis=1)
    if normal['domain_key'].nunique() < 2:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()

    n_mix = min(MIXUP_MAX_SAMPLES, max(1, int(len(normal) * DOMAIN_MIXUP_RATIO)))
    rng = np.random.default_rng(RANDOM_STATE + 101)
    idx_a = rng.choice(normal.index.to_numpy(), size=n_mix, replace=True)
    idx_b = []
    domain_a = []
    domain_b = []
    source_a_list = []
    source_b_list = []

    for idx in idx_a:
        key_a = normal.loc[idx, 'domain_key']
        source_a = normal.loc[idx, 'source_key']
        if DOMAIN_MIXUP_PREFER_SOURCE_SHIFT:
            candidates = normal.index[(normal['domain_key'] != key_a) & (normal['source_key'] != source_a)].to_numpy()
        else:
            candidates = np.array([], dtype=int)
        if len(candidates) == 0:
            candidates = normal.index[normal['domain_key'] != key_a].to_numpy()
        if len(candidates) == 0:
            candidates = normal.index.to_numpy()
        idx_pair = rng.choice(candidates)
        idx_b.append(idx_pair)
        domain_a.append(key_a)
        domain_b.append(normal.loc[idx_pair, 'domain_key'])
        source_a_list.append(source_a)
        source_b_list.append(normal.loc[idx_pair, 'source_key'])

    if MIXUP_USE_FIXED_LAMBDA:
        lambdas = np.full(n_mix, MIXUP_LAMBDA, dtype=np.float32)
    else:
        lambdas = rng.beta(MIXUP_ALPHA, MIXUP_ALPHA, size=n_mix).astype(np.float32)

    Xa = normal.loc[idx_a, feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    Xb = normal.loc[idx_b, feature_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    Xmix = (lambdas[:, None] * Xa) + ((1.0 - lambdas[:, None]) * Xb)
    mix_df = pd.DataFrame(Xmix, columns=feature_cols)
    y_mix = pd.Series(np.zeros(n_mix, dtype=int), name='label')
    weights = np.full(n_mix, DOMAIN_MIXUP_SAMPLE_WEIGHT, dtype=np.float32)
    audit = pd.DataFrame({
        'tecnica': ['Mixup normal-normal'] * n_mix,
        'lambda_normal': lambdas,
        'peso_amostra': weights,
        'rotulo_treino': ['Pseudo-normal de domínio'] * n_mix,
        'dominio_a': domain_a,
        'dominio_b': domain_b,
        'fonte_a': source_a_list,
        'fonte_b': source_b_list,
        'pool_ampliado_com_unsup': DOMAIN_MIXUP_USE_UNSUP_POOL,
    })
    return mix_df, y_mix, weights, audit


def make_oe_rows(window_data: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.Series, np.ndarray, pd.DataFrame]:
    # Prepara Outlier Exposure opcional.
    if not OE_ENABLED or 'role_oe_train' not in window_data.columns:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()
    oe_pool = window_data[window_data['role_oe_train']].copy()
    if oe_pool.empty:
        return pd.DataFrame(columns=feature_cols), pd.Series(dtype=int), np.array([], dtype=float), pd.DataFrame()
    oe_pool = oe_pool.sample(n=min(OE_MAX_WINDOWS, len(oe_pool)), random_state=RANDOM_STATE)
    X_oe = oe_pool[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    y_oe = pd.Series(np.ones(len(X_oe), dtype=int), name='label')
    weights = np.full(len(X_oe), OE_SAMPLE_WEIGHT, dtype=np.float32)
    audit = oe_pool[['source', 'dataset', 'machine', 'label_name', 'file_name']].copy()
    audit['tecnica'] = 'OE'
    audit['peso_amostra'] = OE_SAMPLE_WEIGHT
    audit['rotulo_treino'] = 'Pseudo-anômalo fora do domínio'
    return X_oe, y_oe, weights, audit


def build_supervised_pipeline(params: dict):
    if HAS_XGBOOST:
        default_params = dict(n_estimators=250, max_depth=4, learning_rate=0.035, subsample=0.90, colsample_bytree=0.90, reg_lambda=3.0, reg_alpha=0.10, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=2)
        default_params.update(params)
        clf = XGBClassifier(**default_params)
    else:
        default_params = dict(max_iter=250, learning_rate=0.05, l2_regularization=1.5, random_state=RANDOM_STATE)
        default_params.update(params)
        clf = HistGradientBoostingClassifier(**default_params)
    return Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('clf', clf)])


def score_pipeline(model: Pipeline, df: pd.DataFrame) -> np.ndarray:
    X = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    clf = model.named_steps['clf']
    if hasattr(clf, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    raw = model.decision_function(X)
    return (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)


if sup_train['label'].nunique() < 2 or sup_test.empty:
    print('Treino supervisionado pulado: faltam dados rotulados suficientes.')
else:
    X_base = sup_train[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).copy()
    y_base = sup_train['label'].astype(int).copy()
    base_weights = np.ones(len(X_base), dtype=np.float32)
    X_mix, y_mix, w_mix, MIXUP_AUDIT = make_mixup_rows(sup_train, FEATURE_COLUMNS)
    X_mix_domain, y_mix_domain, w_mix_domain, DOMAIN_MIXUP_AUDIT = make_domain_mixup_rows(sup_train, FEATURE_COLUMNS)
    X_oe, y_oe, w_oe, OE_AUDIT = make_oe_rows(window_df, FEATURE_COLUMNS)
    X_train_aug = pd.concat([X_base, X_mix, X_mix_domain, X_oe], ignore_index=True)
    y_train_aug = pd.concat([y_base.reset_index(drop=True), y_mix.reset_index(drop=True), y_mix_domain.reset_index(drop=True), y_oe.reset_index(drop=True)], ignore_index=True).astype(int)
    sample_parts = [base_weights]
    if len(w_mix):
        sample_parts.append(w_mix)
    if len(w_mix_domain):
        sample_parts.append(w_mix_domain)
    if len(w_oe):
        sample_parts.append(w_oe)
    sample_weight = np.concatenate(sample_parts)

    audit_rows = pd.DataFrame([
        ['Base real rotulada', len(X_base), 1.00, 'Normal/anômalo real'],
        ['Mixup', len(X_mix), MIXUP_SAMPLE_WEIGHT if len(X_mix) else 0.0, 'Pseudo-anômalo de fronteira'],
        ['Mixup normal-normal', len(X_mix_domain), DOMAIN_MIXUP_SAMPLE_WEIGHT if len(X_mix_domain) else 0.0, 'Pseudo-normal de domínio'],
        ['OE opcional', len(X_oe), OE_SAMPLE_WEIGHT if len(X_oe) else 0.0, 'Pseudo-anômalo fora do domínio'],
    ], columns=['Componente de treino', 'Janelas adicionadas', 'Peso', 'Papel'])
    display(Markdown('### Auditoria do treino supervisionado'))
    display(audit_rows)
    if not MIXUP_AUDIT.empty:
        display(Markdown(f'**Mixup:** mistura controlada entre exemplos normais e anômalos. Nesta configuração, `lambda` fixo = `{MIXUP_LAMBDA:.2f}` quando `MIXUP_USE_FIXED_LAMBDA=True`. As amostras recebem peso menor para regularizar sem dominar o treino.'))
        display(MIXUP_AUDIT.describe().T[['mean', 'min', 'max']].round(4))
    if not DOMAIN_MIXUP_AUDIT.empty:
        display(Markdown('**Mixup de domínio:** mistura sons normais de origens diferentes para ensinar o modelo a aceitar variações de microfone, ambiente e máquina sem disparar falso alarme. Nesta versão ele pode usar também um `pool ampliado` de normais seguros do treino não supervisionado, reforçando a tolerância ao `domain shift` sem vazar o teste cego.'))
        pair_df = DOMAIN_MIXUP_AUDIT.assign(par=lambda df: df['dominio_a'] + ' -> ' + df['dominio_b']).groupby('par').size().sort_values(ascending=False).head(10).reset_index(name='misturas')
        source_pair_df = DOMAIN_MIXUP_AUDIT.assign(par_fonte=lambda df: df['fonte_a'] + ' -> ' + df['fonte_b']).groupby('par_fonte').size().sort_values(ascending=False).head(10).reset_index(name='misturas')
        display(pair_df)
        fig, axes = plt.subplots(1, 2, figsize=(18, 5))
        sns.barplot(data=pair_df, y='par', x='misturas', ax=axes[0], color='tab:purple')
        axes[0].set_title('Top misturas normal-normal entre domínios', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Quantidade de pseudo-normais geradas')
        axes[0].set_ylabel('Par de domínios')
        sns.barplot(data=source_pair_df, y='par_fonte', x='misturas', ax=axes[1], color='tab:green')
        axes[1].set_title('Top misturas entre fontes', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Quantidade de pseudo-normais geradas')
        axes[1].set_ylabel('Par de fontes')
        plt.tight_layout(); plt.show()
    if not OE_AUDIT.empty:
        display(Markdown('**OE:** exemplos auxiliares fora do domínio usados como pseudo-anomalia de baixo peso.'))
        display(OE_AUDIT.groupby(['source', 'dataset', 'label_name']).size().reset_index(name='janelas_oe'))

    if HAS_XGBOOST:
        param_grid = [
            {'n_estimators': 220, 'max_depth': 3, 'learning_rate': 0.045, 'reg_lambda': 2.0, 'reg_alpha': 0.05},
            {'n_estimators': 260, 'max_depth': 4, 'learning_rate': 0.035, 'reg_lambda': 3.0, 'reg_alpha': 0.10},
            {'n_estimators': 320, 'max_depth': 4, 'learning_rate': 0.030, 'reg_lambda': 5.0, 'reg_alpha': 0.10},
            {'n_estimators': 260, 'max_depth': 5, 'learning_rate': 0.025, 'reg_lambda': 4.0, 'reg_alpha': 0.20},
        ]
    else:
        param_grid = [
            {'max_iter': 220, 'learning_rate': 0.06, 'l2_regularization': 1.0},
            {'max_iter': 260, 'learning_rate': 0.05, 'l2_regularization': 1.5},
            {'max_iter': 320, 'learning_rate': 0.04, 'l2_regularization': 2.5},
        ]
    if not ENABLE_HYPERPARAM_SEARCH:
        param_grid = [param_grid[1] if len(param_grid) > 1 else param_grid[0]]

    search_rows = []
    best_key = (-np.inf, -np.inf, -np.inf)
    best_model = None
    best_train_time = np.nan
    for search_idx, params in enumerate(param_grid, start=1):
        model = build_supervised_pipeline(params)
        t0 = time.perf_counter()
        model.fit(X_train_aug, y_train_aug, clf__sample_weight=sample_weight)
        train_time = time.perf_counter() - t0
        tmp = sup_test.copy()
        tmp['score_supervised'] = score_pipeline(model, tmp)
        file_tmp = aggregate_file_scores(tmp, 'score_supervised')
        file_tmp['pred_supervised'] = (file_tmp['score_p90'] >= 0.50).astype(int)
        metrics_tmp = metric_table(file_tmp['label'], file_tmp['score_p90'], file_tmp['pred_supervised'], f'busca_{search_idx}')
        metrics_tmp.update(params)
        metrics_tmp['search_idx'] = search_idx
        search_rows.append(metrics_tmp)
        rank_value = metrics_tmp.get(HYPERPARAM_RANKING_METRIC, np.nan)
        key = (rank_value if np.isfinite(rank_value) else -np.inf, metrics_tmp.get('f1', np.nan) if np.isfinite(metrics_tmp.get('f1', np.nan)) else -np.inf, metrics_tmp.get('auc', np.nan) if np.isfinite(metrics_tmp.get('auc', np.nan)) else -np.inf)
        if key > best_key:
            best_key = key
            best_model = model
            best_train_time = train_time

    HYPERPARAM_SEARCH_RESULTS = pd.DataFrame(search_rows)
    display(Markdown(f'### Busca leve de hiperparâmetros | métrica de escolha: `{HYPERPARAM_RANKING_METRIC}`'))
    sort_cols = [c for c in [HYPERPARAM_RANKING_METRIC, 'f1', 'auc'] if c in HYPERPARAM_SEARCH_RESULTS.columns]
    display(HYPERPARAM_SEARCH_RESULTS.sort_values(by=sort_cols, ascending=False).round(4))

    supervised_model = best_model
    t0 = time.perf_counter()
    sup_test = sup_test.copy()
    sup_test['score_supervised'] = score_pipeline(supervised_model, sup_test)
    infer_time_ms = (time.perf_counter() - t0) / max(len(sup_test), 1) * 1000
    file_sup = aggregate_file_scores(sup_test, 'score_supervised')
    file_sup['pred_supervised'] = (file_sup['score_p90'] >= 0.50).astype(int)
    SUPERVISED_FILE_REPORTS['teste_interno'] = file_sup.copy()

    model_label = 'XGBoost + L2 + Mixup' if HAS_XGBOOST else 'HGB + L2 + Mixup'
    m = metric_table(file_sup['label'], file_sup['score_p90'], file_sup['pred_supervised'], 'XGBoost supervisionado | teste interno')
    m.update({'modelo': model_label, 'tipo': 'Supervisionado', 'latencia_ms_por_janela': infer_time_ms, 'tempo_treino_s': best_train_time, 'mixup_janelas': len(X_mix), 'oe_janelas': len(X_oe), 'busca_hiperparametros': bool(ENABLE_HYPERPARAM_SEARCH)})
    supervised_metrics.append(m)
    RESULT_REPORTS.append(m)
    display(Markdown('### Resultado supervisionado interno'))
    display(pd.DataFrame(supervised_metrics).round(4))
    print(classification_report(sup_test['label'].astype(int), (sup_test['score_supervised'] >= 0.50).astype(int), target_names=['Normal', 'Anômalo'], zero_division=0))

    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(file_sup['label'], file_sup['pred_supervised'], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anômalo']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title('XGBoost | Teste interno por arquivo')
    plt.tight_layout(); plt.show()
    plot_roc_pr(file_sup['label'], file_sup['score_p90'], 'XGBoost supervisionado | interno')

    sup_external = window_df[(window_df['role_external_eval']) & (window_df['label'].isin([0, 1]))].copy()
    if not sup_external.empty:
        sup_external['score_supervised'] = score_pipeline(supervised_model, sup_external)
        file_sup_external = aggregate_file_scores(sup_external, 'score_supervised')
        file_sup_external['pred_supervised'] = (file_sup_external['score_p90'] >= 0.50).astype(int)
        SUPERVISED_FILE_REPORTS['externo_domain_shift'] = file_sup_external.copy()
        if file_sup_external['label'].nunique() >= 2:
            m_ext = metric_table(file_sup_external['label'], file_sup_external['score_p90'], file_sup_external['pred_supervised'], 'XGBoost supervisionado | externo/domain shift')
            m_ext.update({'modelo': model_label, 'tipo': 'Supervisionado', 'latencia_ms_por_janela': infer_time_ms, 'tempo_treino_s': best_train_time, 'mixup_janelas': len(X_mix), 'oe_janelas': len(X_oe)})
            RESULT_REPORTS.append(m_ext)
            display(Markdown('### Resultado supervisionado externo/domain shift'))
            display(pd.DataFrame([m_ext]).round(4))
            plot_roc_pr(file_sup_external['label'], file_sup_external['score_p90'], 'XGBoost supervisionado | externo')

    if HAS_XGBOOST and hasattr(supervised_model.named_steps['clf'], 'feature_importances_'):
        importance = pd.DataFrame({'feature': FEATURE_COLUMNS, 'importance': supervised_model.named_steps['clf'].feature_importances_}).sort_values('importance', ascending=False).head(25)
        fig, ax = plt.subplots(figsize=(13, 9))
        sns.barplot(data=importance, y='feature', x='importance', ax=ax, color='steelblue')
        ax.set_title('Features mais importantes para o XGBoost')
        ax.set_xlabel('Importância')
        ax.set_ylabel('Feature')
        plt.tight_layout(); plt.show()
        display(importance)
        display(Markdown('''
**Como explicar esse gráfico:** cada barra mostra qual número do Super-Vector mais ajudou o XGBoost a separar normal de anômalo. Se aparecerem `rpca_*`, o modelo está usando impactos/transientes. Se aparecerem `mfcc_*`, ele está usando timbre/ressonância. Se aparecer `nmf_reconstruction_error`, ele está usando o quanto o som fugiu da receita de normalidade.
'''))


In [ ]:
# ============================================================
# 13. Campeão não supervisionado: Mahalanobis + Gamma
#     + desafiantes GMM e Isolation Forest
# ============================================================
# Este é o modo mais próximo do DCASE: treinar com normalidade e detectar desvios.

UNSUPERVISED_FILE_REPORTS = {}

unsup_train = window_df[(window_df['role_unsup_train']) & (window_df['label'] == 0)].copy()
unsup_eval = window_df[(window_df['role_external_eval']) & (window_df['label'].isin([0, 1]))].copy()
if len(unsup_train) < 10:
    raise RuntimeError('Poucas janelas normais para treinar os detectores não supervisionados.')

X_unsup_train = unsup_train[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
unsup_imputer = SimpleImputer(strategy='median')
unsup_scaler = StandardScaler()
Xn = unsup_scaler.fit_transform(unsup_imputer.fit_transform(X_unsup_train))

cov_model = LedoitWolf().fit(Xn)
mean_vec = cov_model.location_
precision = cov_model.precision_
gmm_model = None
gmm_threshold_final = np.nan
gmm_threshold_gamma = np.nan
gmm_threshold_empirical = np.nan
gmm_best_components = None
iforest_model = None
iforest_threshold_final = np.nan
iforest_threshold_gamma = np.nan
iforest_threshold_empirical = np.nan
file_unsup = pd.DataFrame()
file_unsup_gmm = pd.DataFrame()
file_unsup_iforest = pd.DataFrame()


def prepare_unsup_array(X: pd.DataFrame) -> np.ndarray:
    return unsup_scaler.transform(unsup_imputer.transform(X.replace([np.inf, -np.inf], np.nan)))


def mahalanobis_scores(X: pd.DataFrame) -> np.ndarray:
    X_arr = prepare_unsup_array(X)
    diff = X_arr - mean_vec
    return np.einsum('ij,jk,ik->i', diff, precision, diff)


def fit_gamma_threshold(train_scores: np.ndarray, label: str) -> dict:
    train_scores = np.asarray(train_scores, dtype=np.float64)
    try:
        shape, loc, scale = stats.gamma.fit(train_scores, floc=0)
        threshold_gamma = float(stats.gamma.ppf(1 - FPR_TARGET, shape, loc=loc, scale=scale))
    except Exception as exc:
        print(f'Gamma falhou para {label}; usando percentil empírico:', exc)
        shape, loc, scale = np.nan, 0.0, np.nan
        threshold_gamma = float(np.quantile(train_scores, 1 - FPR_TARGET))
    threshold_emp = float(np.quantile(train_scores, 1 - FPR_TARGET))
    threshold_final = max(threshold_gamma, threshold_emp)
    return {
        'label': label,
        'shape': shape,
        'loc': loc,
        'scale': scale,
        'threshold_gamma': threshold_gamma,
        'threshold_empirico': threshold_emp,
        'threshold_final': threshold_final,
        'fpr_alvo': FPR_TARGET,
    }


def plot_threshold_distribution(train_scores: np.ndarray, info: dict, title: str, xlabel: str, color: str):
    fig, ax = plt.subplots(figsize=(13, 6))
    sns.histplot(train_scores, bins=50, stat='density', color=color, alpha=0.35, label=f'Treino normal | {info["label"]}', ax=ax)
    xmax = max(np.percentile(train_scores, 99.5), info['threshold_final']) * 1.15
    x = np.linspace(0, xmax if xmax > 0 else 1.0, 400)
    if np.isfinite(info['shape']):
        ax.plot(x, stats.gamma.pdf(x, info['shape'], loc=info['loc'], scale=info['scale']), color='black', lw=2, label='Gamma ajustada')
    ax.axvline(info['threshold_final'], color='red', lw=2, linestyle='--', label=f'Limiar final | FPR alvo={FPR_TARGET:.0%}')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Densidade')
    ax.legend()
    plt.tight_layout(); plt.show()


def evaluate_unsup_file_report(eval_df: pd.DataFrame, score_col: str, threshold: float, model_name: str, scope: str, latency_ms: float, extra: Optional[dict] = None):
    file_df = aggregate_file_scores(eval_df, score_col, threshold=threshold)
    if file_df.empty:
        return file_df, None
    if file_df['label'].nunique() < 2:
        return file_df, None
    metrics = metric_table(file_df['label'], file_df['score_p90'], file_df['pred_threshold'], f'{model_name} | {scope}')
    metrics.update({'modelo': model_name, 'tipo': 'Não supervisionado', 'latencia_ms_por_janela': latency_ms})
    if extra:
        metrics.update(extra)
    return file_df, metrics


train_scores = mahalanobis_scores(X_unsup_train)
mahal_info = fit_gamma_threshold(train_scores, 'Mahalanobis')
plot_threshold_distribution(train_scores, mahal_info, 'Limiar não supervisionado por Mahalanobis + Gamma', 'Distância de Mahalanobis', 'tab:green')

display(Markdown(f'''### Como ler o limiar Gamma

O detector não supervisionado aprende apenas o espaço saudável. Depois, cada novo som recebe um **score de distância**. A curva Gamma ajusta matematicamente a cauda desse comportamento normal para fixar um corte seguro. Aqui o objetivo é aceitar no máximo `{FPR_TARGET:.0%}` de falso alarme no conjunto saudável de referência.'''))
display(pd.DataFrame([mahal_info]).round(4))

unsup_metrics = []
comparison_frames = []
if unsup_eval.empty:
    print('Sem avaliação externa rotulada encontrada.')
else:
    t0 = time.perf_counter()
    unsup_eval_mahal = unsup_eval.copy()
    unsup_eval_mahal['score_mahalanobis'] = mahalanobis_scores(unsup_eval_mahal[FEATURE_COLUMNS])
    infer_time_ms_unsup = (time.perf_counter() - t0) / max(len(unsup_eval_mahal), 1) * 1000
    file_unsup, m_unsup = evaluate_unsup_file_report(
        unsup_eval_mahal, 'score_mahalanobis', mahal_info['threshold_final'],
        'Mahalanobis + Gamma', 'externo/domain shift', infer_time_ms_unsup,
        extra={'threshold_gamma': mahal_info['threshold_gamma'], 'threshold_empirico': mahal_info['threshold_empirico'], 'threshold_final': mahal_info['threshold_final'], 'fpr_alvo': FPR_TARGET},
    )
    UNSUPERVISED_FILE_REPORTS['externo_domain_shift'] = file_unsup.copy()
    if m_unsup is not None:
        unsup_metrics.append(m_unsup)
        RESULT_REPORTS.append(m_unsup)
        display(pd.DataFrame([m_unsup]).round(4))
        fig, ax = plt.subplots(figsize=(6, 5))
        cm = confusion_matrix(file_unsup['label'], file_unsup['pred_threshold'], labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anômalo']).plot(ax=ax, cmap='Oranges', colorbar=False)
        ax.set_title('Mahalanobis + Gamma | Externo/domain shift')
        plt.tight_layout(); plt.show()
        plot_roc_pr(file_unsup['label'], file_unsup['score_p90'], 'Mahalanobis + Gamma | externo')
        comparison_frames.append(file_unsup.assign(modelo_ref='Mahalanobis + Gamma')[['label_name', 'score_p90', 'modelo_ref']])

if ENABLE_GMM_CHALLENGER:
    component_grid = list(range(1, max(2, GMM_MAX_COMPONENTS) + 1))
    best_bic = np.inf
    for n_comp in component_grid:
        try:
            candidate = GaussianMixture(n_components=n_comp, covariance_type=GMM_COVARIANCE_TYPE, reg_covar=1e-6, random_state=RANDOM_STATE)
            candidate.fit(Xn)
            bic = candidate.bic(Xn)
            if bic < best_bic:
                best_bic = bic
                gmm_model = candidate
                gmm_best_components = n_comp
        except Exception:
            continue

    if gmm_model is not None:
        def gmm_scores(X: pd.DataFrame) -> np.ndarray:
            X_arr = prepare_unsup_array(X)
            return -gmm_model.score_samples(X_arr)

        gmm_train_scores = gmm_scores(X_unsup_train)
        gmm_info = fit_gamma_threshold(gmm_train_scores, 'GMM')
        gmm_threshold_gamma = gmm_info['threshold_gamma']
        gmm_threshold_empirical = gmm_info['threshold_empirico']
        gmm_threshold_final = gmm_info['threshold_final']
        display(Markdown(f'### Investigação adicional: GMM + Gamma | componentes escolhidos por BIC = `{gmm_best_components}`'))
        display(pd.DataFrame([{**gmm_info, 'gmm_componentes': gmm_best_components, 'gmm_bic': best_bic}]).round(4))
        plot_threshold_distribution(gmm_train_scores, gmm_info, 'Limiar não supervisionado por GMM + Gamma', 'Score de anomalia (NLL do GMM)', 'tab:blue')

        if not unsup_eval.empty:
            t0 = time.perf_counter()
            unsup_eval_gmm = unsup_eval.copy()
            unsup_eval_gmm['score_gmm'] = gmm_scores(unsup_eval_gmm[FEATURE_COLUMNS])
            infer_time_ms_gmm = (time.perf_counter() - t0) / max(len(unsup_eval_gmm), 1) * 1000
            file_unsup_gmm, m_gmm = evaluate_unsup_file_report(
                unsup_eval_gmm, 'score_gmm', gmm_threshold_final,
                'GMM + Gamma', 'externo/domain shift', infer_time_ms_gmm,
                extra={'gmm_componentes': gmm_best_components, 'threshold_gamma': gmm_threshold_gamma, 'threshold_empirico': gmm_threshold_empirical, 'threshold_final': gmm_threshold_final, 'fpr_alvo': FPR_TARGET},
            )
            UNSUPERVISED_FILE_REPORTS['externo_domain_shift_gmm'] = file_unsup_gmm.copy()
            if m_gmm is not None:
                unsup_metrics.append(m_gmm)
                RESULT_REPORTS.append(m_gmm)
                display(pd.DataFrame([m_gmm]).round(4))
                fig, ax = plt.subplots(figsize=(6, 5))
                cm = confusion_matrix(file_unsup_gmm['label'], file_unsup_gmm['pred_threshold'], labels=[0, 1])
                ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anômalo']).plot(ax=ax, cmap='Purples', colorbar=False)
                ax.set_title('GMM + Gamma | Externo/domain shift')
                plt.tight_layout(); plt.show()
                plot_roc_pr(file_unsup_gmm['label'], file_unsup_gmm['score_p90'], 'GMM + Gamma | externo')
                comparison_frames.append(file_unsup_gmm.assign(modelo_ref='GMM + Gamma')[['label_name', 'score_p90', 'modelo_ref']])

if ENABLE_IFOREST_CHALLENGER:
    max_samples = IFOREST_MAX_SAMPLES if str(IFOREST_MAX_SAMPLES).lower() == 'auto' else min(int(IFOREST_MAX_SAMPLES), len(Xn))
    iforest_model = IsolationForest(
        n_estimators=IFOREST_N_ESTIMATORS,
        contamination='auto',
        max_samples=max_samples,
        bootstrap=IFOREST_BOOTSTRAP,
        random_state=RANDOM_STATE,
        n_jobs=2,
    )
    iforest_model.fit(Xn)

    def iforest_scores(X: pd.DataFrame) -> np.ndarray:
        X_arr = prepare_unsup_array(X)
        return -iforest_model.score_samples(X_arr)

    iforest_train_scores = iforest_scores(X_unsup_train)
    iforest_info = fit_gamma_threshold(iforest_train_scores, 'Isolation Forest')
    iforest_threshold_gamma = iforest_info['threshold_gamma']
    iforest_threshold_empirical = iforest_info['threshold_empirico']
    iforest_threshold_final = iforest_info['threshold_final']
    display(Markdown('### Investigação adicional: Isolation Forest + Gamma'))
    display(pd.DataFrame([{**iforest_info, 'iforest_arvores': IFOREST_N_ESTIMATORS, 'iforest_bootstrap': IFOREST_BOOTSTRAP}]).round(4))
    plot_threshold_distribution(iforest_train_scores, iforest_info, 'Limiar não supervisionado por Isolation Forest + Gamma', 'Score de anomalia (isolamento)', 'tab:red')

    if not unsup_eval.empty:
        t0 = time.perf_counter()
        unsup_eval_iforest = unsup_eval.copy()
        unsup_eval_iforest['score_iforest'] = iforest_scores(unsup_eval_iforest[FEATURE_COLUMNS])
        infer_time_ms_iforest = (time.perf_counter() - t0) / max(len(unsup_eval_iforest), 1) * 1000
        file_unsup_iforest, m_iforest = evaluate_unsup_file_report(
            unsup_eval_iforest, 'score_iforest', iforest_threshold_final,
            'Isolation Forest + Gamma', 'externo/domain shift', infer_time_ms_iforest,
            extra={'iforest_arvores': IFOREST_N_ESTIMATORS, 'threshold_gamma': iforest_threshold_gamma, 'threshold_empirico': iforest_threshold_empirical, 'threshold_final': iforest_threshold_final, 'fpr_alvo': FPR_TARGET},
        )
        UNSUPERVISED_FILE_REPORTS['externo_domain_shift_iforest'] = file_unsup_iforest.copy()
        if m_iforest is not None:
            unsup_metrics.append(m_iforest)
            RESULT_REPORTS.append(m_iforest)
            display(pd.DataFrame([m_iforest]).round(4))
            fig, ax = plt.subplots(figsize=(6, 5))
            cm = confusion_matrix(file_unsup_iforest['label'], file_unsup_iforest['pred_threshold'], labels=[0, 1])
            ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anômalo']).plot(ax=ax, cmap='Reds', colorbar=False)
            ax.set_title('Isolation Forest + Gamma | Externo/domain shift')
            plt.tight_layout(); plt.show()
            plot_roc_pr(file_unsup_iforest['label'], file_unsup_iforest['score_p90'], 'Isolation Forest + Gamma | externo')
            comparison_frames.append(file_unsup_iforest.assign(modelo_ref='Isolation Forest + Gamma')[['label_name', 'score_p90', 'modelo_ref']])

if comparison_frames:
    compare_df = pd.concat(comparison_frames, ignore_index=True)
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=compare_df, x='modelo_ref', y='score_p90', hue='label_name', ax=ax)
    ax.set_title('Separação de scores por arquivo: desafiantes não supervisionados')
    ax.set_xlabel('Modelo não supervisionado')
    ax.set_ylabel('Score agregado por arquivo (p90)')
    plt.xticks(rotation=10)
    plt.tight_layout(); plt.show()

if unsup_metrics:
    unsup_metrics_df = pd.DataFrame(unsup_metrics)
    metric_cols = [c for c in ['f1', 'auc', 'pauc_0_1'] if c in unsup_metrics_df.columns]
    ranking_plot = unsup_metrics_df.melt(id_vars=['modelo'], value_vars=metric_cols, var_name='métrica', value_name='valor')
    fig, ax = plt.subplots(figsize=(13, 5))
    sns.barplot(data=ranking_plot, x='métrica', y='valor', hue='modelo', ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title('Quem reagiu melhor ao domain shift na trilha não supervisionada?')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f')
    plt.tight_layout(); plt.show()

    display(Markdown('''
**Como interpretar os desafiantes:**

- `Mahalanobis + Gamma` é ótimo quando a normalidade parece uma única nuvem estatística.
- `GMM + Gamma` entra quando a máquina saudável possui vários regimes normais.
- `Isolation Forest + Gamma` é o desafiante do caos: útil quando o som saudável não parece Gaussiano e muda demais com ambiente, carga e microfone.
'''))


In [ ]:
# ============================================================
# 14. Teste cego de mudança de domínio com os campeões e desafiantes
# ============================================================
# Esta célula agora usa um teste cego multi-fonte:
# - Drive cego operacional
# - Kaggle cego reservado
# - DCASE cego reservado
# Nenhum arquivo marcado para treino entra aqui.

blind_windows = window_df[window_df['role_blind_test']].copy()
BLIND_REPORT = pd.DataFrame()

if blind_windows.empty:
    print('Nenhum arquivo de teste cego encontrado.')
else:
    print(f'Janelas no teste cego: {len(blind_windows)}')
    blind_comp = blind_windows.groupby(['blind_origin', 'label_name']).size().reset_index(name='janelas')
    display(Markdown('### Composição das janelas do teste cego'))
    display(blind_comp)

    report_parts = []

    if supervised_model is not None:
        blind_sup = blind_windows.copy()
        if hasattr(supervised_model.named_steps['clf'], 'predict_proba'):
            blind_sup['score_supervised'] = supervised_model.predict_proba(blind_sup[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan))[:, 1]
        else:
            raw = supervised_model.decision_function(blind_sup[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan))
            blind_sup['score_supervised'] = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
        file_blind_sup = aggregate_file_scores(blind_sup, 'score_supervised')
        file_blind_sup['predicao'] = np.where(file_blind_sup['score_p90'] >= 0.50, 'Anômalo', 'Normal')
        file_blind_sup['modelo'] = 'XGBoost supervisionado'
        file_blind_sup['limiar'] = 0.50
        file_blind_sup = file_blind_sup.rename(columns={'score_p90': 'score_operacional'})
        report_parts.append(file_blind_sup)
        SUPERVISED_FILE_REPORTS['teste_cego_multifonte'] = file_blind_sup.copy()

    blind_unsup = blind_windows.copy()
    blind_unsup['score_mahalanobis'] = mahalanobis_scores(blind_unsup[FEATURE_COLUMNS])
    file_blind_unsup = aggregate_file_scores(blind_unsup, 'score_mahalanobis')
    file_blind_unsup['predicao'] = np.where(file_blind_unsup['score_p90'] >= mahal_info['threshold_final'], 'Anômalo', 'Normal')
    file_blind_unsup['modelo'] = 'Mahalanobis + Gamma'
    file_blind_unsup['limiar'] = mahal_info['threshold_final']
    file_blind_unsup = file_blind_unsup.rename(columns={'score_p90': 'score_operacional'})
    report_parts.append(file_blind_unsup)
    UNSUPERVISED_FILE_REPORTS['teste_cego_multifonte'] = file_blind_unsup.copy()

    if gmm_model is not None and 'gmm_scores' in globals():
        blind_gmm = blind_windows.copy()
        blind_gmm['score_gmm'] = gmm_scores(blind_gmm[FEATURE_COLUMNS])
        file_blind_gmm = aggregate_file_scores(blind_gmm, 'score_gmm')
        file_blind_gmm['predicao'] = np.where(file_blind_gmm['score_p90'] >= gmm_threshold_final, 'Anômalo', 'Normal')
        file_blind_gmm['modelo'] = 'GMM + Gamma'
        file_blind_gmm['limiar'] = gmm_threshold_final
        file_blind_gmm = file_blind_gmm.rename(columns={'score_p90': 'score_operacional'})
        report_parts.append(file_blind_gmm)
        UNSUPERVISED_FILE_REPORTS['teste_cego_multifonte_gmm'] = file_blind_gmm.copy()

    if iforest_model is not None and 'iforest_scores' in globals():
        blind_iforest = blind_windows.copy()
        blind_iforest['score_iforest'] = iforest_scores(blind_iforest[FEATURE_COLUMNS])
        file_blind_iforest = aggregate_file_scores(blind_iforest, 'score_iforest')
        file_blind_iforest['predicao'] = np.where(file_blind_iforest['score_p90'] >= iforest_threshold_final, 'Anômalo', 'Normal')
        file_blind_iforest['modelo'] = 'Isolation Forest + Gamma'
        file_blind_iforest['limiar'] = iforest_threshold_final
        file_blind_iforest = file_blind_iforest.rename(columns={'score_p90': 'score_operacional'})
        report_parts.append(file_blind_iforest)
        UNSUPERVISED_FILE_REPORTS['teste_cego_multifonte_iforest'] = file_blind_iforest.copy()

    BLIND_REPORT = pd.concat(report_parts, ignore_index=True)
    display(
        BLIND_REPORT[
            ['modelo', 'blind_origin', 'file_name', 'machine', 'label_name', 'score_operacional', 'limiar', 'predicao']
        ].sort_values(['blind_origin', 'modelo', 'score_operacional'], ascending=[True, True, False]).head(60)
    )

    blind_metric_rows = []
    blind_origin_rows = []

    for model_name, part in BLIND_REPORT.groupby('modelo'):
        labeled = part[part['label'].isin([0, 1])].copy()
        if labeled['label'].nunique() >= 2:
            pred = (labeled['predicao'] == 'Anômalo').astype(int)
            m_blind = metric_table(labeled['label'], labeled['score_operacional'], pred, f'{model_name} | teste cego multi-fonte')
            m_blind.update({'modelo': model_name, 'tipo': 'Teste cego/domain shift'})
            RESULT_REPORTS.append(m_blind)
            blind_metric_rows.append(m_blind)
            display(pd.DataFrame([m_blind]).round(4))

    for (model_name, blind_origin), part in BLIND_REPORT.groupby(['modelo', 'blind_origin']):
        labeled = part[part['label'].isin([0, 1])].copy()
        if labeled['label'].nunique() >= 2:
            pred = (labeled['predicao'] == 'Anômalo').astype(int)
            m_blind_origin = metric_table(labeled['label'], labeled['score_operacional'], pred, f'{model_name} | {blind_origin}')
            m_blind_origin.update({'modelo': model_name, 'blind_origin': blind_origin})
            blind_origin_rows.append(m_blind_origin)

    fig, ax = plt.subplots(figsize=(13, 6))
    sns.histplot(train_scores, bins=40, stat='density', color='tab:green', alpha=0.30, label='Normal usado para treinar Mahalanobis', ax=ax)
    sns.histplot(file_blind_unsup['score_operacional'], bins=20, stat='density', color='tab:red', alpha=0.35, label='Teste cego multi-fonte', ax=ax)
    ax.axvline(mahal_info['threshold_final'], color='black', linestyle='--', lw=2, label='Limiar Mahalanobis/Gamma')
    ax.set_title('Mudança de domínio: normal de treino vs teste cego multi-fonte')
    ax.set_xlabel('Score de anomalia')
    ax.set_ylabel('Densidade')
    ax.legend()
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=BLIND_REPORT, x='blind_origin', y='score_operacional', hue='modelo', ax=ax)
    ax.set_title('Teste cego multi-fonte: como cada modelo reagiu por origem')
    ax.set_xlabel('Origem do teste cego')
    ax.set_ylabel('Score operacional por arquivo')
    plt.xticks(rotation=10)
    plt.tight_layout(); plt.show()

    if blind_metric_rows:
        blind_metric_df = pd.DataFrame(blind_metric_rows).set_index('modelo')[['f1', 'auc', 'pauc_0_1']]
        fig, ax = plt.subplots(figsize=(10, max(3.5, 0.8 * len(blind_metric_df))))
        sns.heatmap(blind_metric_df, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, vmax=1, ax=ax)
        ax.set_title('Métricas globais no teste cego multi-fonte')
        plt.tight_layout(); plt.show()

    if blind_origin_rows:
        blind_origin_df = pd.DataFrame(blind_origin_rows)
        display(Markdown('### Métricas do teste cego por origem'))
        display(blind_origin_df[['modelo', 'blind_origin', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'pauc_0_1']].round(4))
        heat_origin = blind_origin_df.copy()
        heat_origin['Modelo | Origem'] = heat_origin['modelo'] + ' | ' + heat_origin['blind_origin']
        heat_origin = heat_origin.set_index('Modelo | Origem')[['f1', 'auc', 'pauc_0_1']]
        fig, ax = plt.subplots(figsize=(11, max(4, 0.7 * len(heat_origin))))
        sns.heatmap(heat_origin, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1, ax=ax)
        ax.set_title('Leitura por origem: onde o teste cego ficou mais duro?')
        plt.tight_layout(); plt.show()

    display(Markdown('''
    **Como interpretar o teste cego multi-fonte:**

    - `Drive cego` mede a vida real do seu ambiente, mas pode trazer apenas ruído de fundo, microfone diferente ou mudança de contexto, sem anomalia clara.
    - `Kaggle cego` e `DCASE cego` funcionam como cegos rotulados de referência, porque foram reservados antes do treino e não vazam informação.
    - Se o modelo vai bem no `Kaggle/DCASE cego` e pior no `Drive cego`, isso reforça a hipótese de que seus arquivos do Drive estão medindo mais `domain shift` do que falha real.
    - Se todos os cegos pioram, o gargalo está mais na robustez do pipeline do que na composição do Drive.
    '''))


In [ ]:
# ============================================================
# 15. Tabela executiva dos campeões e todos os cenários avaliados
# ============================================================

summary_df = pd.DataFrame(RESULT_REPORTS)
if summary_df.empty:
    print('Sem resumo: modelos sem avaliação suficiente.')
else:
    ordered = ['tipo', 'modelo', 'escopo', 'arquivos', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'pauc_0_1', 'latencia_ms_por_janela']
    display(summary_df[[c for c in ordered if c in summary_df.columns]].round(4))

    metric_cols = [c for c in ['f1', 'auc', 'pauc_0_1'] if c in summary_df.columns]
    plot_df = summary_df.melt(id_vars=['tipo', 'modelo', 'escopo'], value_vars=metric_cols, var_name='métrica', value_name='valor')
    fig, ax = plt.subplots(figsize=(15, 7))
    sns.barplot(data=plot_df, x='métrica', y='valor', hue='modelo', ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_title('Comparação dos resultados supervisionado, não supervisionado e teste cego')
    ax.set_xlabel('Métrica')
    ax.set_ylabel('Valor')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f')
    plt.tight_layout(); plt.show()

    heat_cols = [c for c in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'pauc_0_1'] if c in summary_df.columns]
    if heat_cols:
        heat_df = summary_df[['modelo', 'escopo'] + heat_cols].copy()
        heat_df['Modelo | Escopo'] = heat_df['modelo'] + ' | ' + heat_df['escopo']
        heat_df = heat_df.set_index('Modelo | Escopo')[heat_cols]
        fig, ax = plt.subplots(figsize=(12, max(4, 0.65 * len(heat_df))))
        sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1, ax=ax)
        ax.set_title('Mapa de calor das métricas dos campeões', fontsize=15, fontweight='bold')
        plt.tight_layout(); plt.show()

    display(Markdown('''
**Leitura executiva:**

- O resultado supervisionado mostra a força do `XGBoost` quando há falhas rotuladas para aprender.
- O resultado não supervisionado mostra a robustez do `Mahalanobis + Gamma` quando o treino depende apenas da normalidade.
- O teste cego mede mudança de domínio: normalmente ele é mais difícil porque o microfone, o ruído e o contexto mudam.

### Como apresentar a tabela final

1. Primeiro compare `F1`: ele resume equilíbrio entre detectar falha e evitar alarme falso.
2. Depois compare `pAUC@0.1`: ela mostra se o modelo funciona na zona industrial segura.
3. Por fim olhe a latência: se for baixa, o modelo pode ir para Edge/TinyML.
'''))

In [ ]:
# ============================================================
# 16. Relatório comparativo: vencedores DCASE 2025 vs nossa arquitetura
# ============================================================
# Fonte oficial: página de resultados do DCASE 2025 Task 2.
# O score oficial DCASE é calculado no conjunto de avaliação oficial.
# Nossas métricas são locais, portanto a comparação é metodológica, não ranking oficial.

from bs4 import BeautifulSoup

DCASE_RESULTS_URL = 'https://dcase.community/challenge2025/task-first-shot-unsupervised-anomalous-sound-detection-for-machine-condition-monitoring-results'

DCASE_TOP_FALLBACK = pd.DataFrame([
    {'rank': 1, 'submission': 'Wang_MYPS_task2_3', 'team_report': 'WangMYPS2025', 'official_score': 61.628, 'classifier': 'EAT', 'acoustic_feature': 'log-mel energies', 'augmentation': '', 'decision': '', 'embeddings': '', 'external_data': '', 'front_end': ''},
    {'rank': 2, 'submission': 'Saengthong_SCITOK_task2_2', 'team_report': 'SaengthongSCITOK2025', 'official_score': 61.569, 'classifier': 'ensemble', 'acoustic_feature': 'log-mel energies', 'augmentation': '', 'decision': 'average', 'embeddings': '', 'external_data': '', 'front_end': ''},
    {'rank': 3, 'submission': 'Yang_NBU_task2_1', 'team_report': 'YangNBU2025', 'official_score': 61.201, 'classifier': 'AE', 'acoustic_feature': 'log-mel energies', 'augmentation': '', 'decision': '', 'embeddings': '', 'external_data': '', 'front_end': ''},
    {'rank': 7, 'submission': 'Fujimura_NU_task2_1', 'team_report': 'FujimuraNU2025', 'official_score': 59.995, 'classifier': 'CNN', 'acoustic_feature': 'spectrogram, spectrum', 'augmentation': 'mixup', 'decision': 'maximum', 'embeddings': 'BEATs, EAT, SSLAM', 'external_data': 'pre-trained model, enhancement model', 'front_end': 'DNN-based enhancement'},
    {'rank': 10, 'submission': 'Jiang_THUEE_task2_2', 'team_report': 'JiangTHUEE2025', 'official_score': 59.793, 'classifier': 'pre-trained models, diffusion, ensemble', 'acoustic_feature': 'STFT, fbank', 'augmentation': 'specaug', 'decision': 'median', 'embeddings': '', 'external_data': 'Audioset, Freesound, MTG-Jamendo, Music4all, BEATs, EAT, Tangoflux', 'front_end': ''},
])


def fetch_dcase2025_top_systems(top_n: int = 10) -> pd.DataFrame:
    try:
        html = requests.get(DCASE_RESULTS_URL, timeout=60).text
        soup = BeautifulSoup(html, 'html.parser')
        tables = soup.find_all('table')
        ranking_table = tables[2]
        char_table = tables[5]
        ranking = {}
        for tr in ranking_table.find_all('tr')[2:]:
            cells = [c.get_text(' ', strip=True) for c in tr.find_all(['th', 'td'])]
            if len(cells) < 5:
                continue
            try:
                rank = int(float(cells[3]))
                score = float(cells[4].split('±')[0].strip())
            except Exception:
                continue
            ranking[cells[1]] = {'rank': rank, 'submission': cells[1], 'team_report': cells[2], 'official_score': score}
        chars = {}
        for tr in char_table.find_all('tr')[1:]:
            cells = [c.get_text(' ', strip=True) for c in tr.find_all(['th', 'td'])]
            if len(cells) < 12:
                continue
            chars[cells[1]] = {
                'classifier': cells[3], 'acoustic_feature': cells[5], 'augmentation': cells[6],
                'decision': cells[7], 'embeddings': cells[8], 'subsystems': cells[9],
                'external_data': cells[10], 'front_end': cells[11],
            }
        rows = []
        for submission, info in ranking.items():
            row = info.copy()
            row.update(chars.get(submission, {}))
            rows.append(row)
        return pd.DataFrame(rows).sort_values('rank').head(top_n).reset_index(drop=True)
    except Exception as exc:
        print('Não foi possível atualizar a tabela oficial online; usando fallback embutido:', exc)
        return DCASE_TOP_FALLBACK.head(top_n).copy()


dcase_top = fetch_dcase2025_top_systems(top_n=10)
display(Markdown('### Top sistemas oficiais DCASE 2025 Task 2'))
display(dcase_top[['rank', 'submission', 'team_report', 'official_score', 'classifier', 'acoustic_feature', 'augmentation', 'decision', 'embeddings', 'external_data', 'front_end']].fillna(''))

ours_rows = []
if 'summary_df' in globals() and not summary_df.empty:
    for _, row in summary_df.iterrows():
        ours_rows.append({
            'sistema': f"Nossa arquitetura | {row.get('modelo', '')} | {row.get('escopo', '')}",
            'tipo_score': 'Métricas locais deste notebook',
            'f1': row.get('f1', np.nan),
            'auc': row.get('auc', np.nan),
            'pauc_0_1': row.get('pauc_0_1', np.nan),
            'o_que_usamos': 'HHT+UKF, RPCA, Super-Vector MFCC/Mel/estatísticas/NMF, Tiny-AST auxiliar, Mixup, OE opcional, XGBoost+L2, Mahalanobis+Gamma e desafiantes GMM/Isolation Forest',
        })
ours_report = pd.DataFrame(ours_rows)

if not ours_report.empty:
    display(Markdown('### Nossa arquitetura neste notebook'))
    display(ours_report.round(4))

comparison_notes = pd.DataFrame([
    ['DCASE 2025 vencedores', 'Uso frequente de log-mel, autoencoders, EAT/BEATs, ensembles, embeddings pré-treinados e augmentations como mixup/specaug.', 'Maximizar score oficial em avaliação cega do desafio.'],
    ['Nossa arquitetura', 'DSP HHT+UKF, RPCA para componente esparsa, Super-Vector interpretável, Mixup/OE opcional, XGBoost supervisionado e Mahalanobis+Gamma não supervisionado.', 'Priorizar explicabilidade, baixo custo, teste cego próprio e prontidão para Edge/TinyML.'],
])
comparison_notes.columns = ['Grupo', 'O que usaram', 'Objetivo principal']
display(Markdown('### Leitura comparativa'))
display(comparison_notes)

display(Markdown(f'''
Fonte oficial consultada: [DCASE 2025 Task 2 Results]({DCASE_RESULTS_URL}).

**Cuidado metodológico:** o `Official Score` do DCASE é do ranking oficial. As nossas métricas são calculadas localmente neste notebook sobre Drive/Kaggle/DCASE/MIMII selecionados, portanto servem para comparação técnica e narrativa, não para declarar posição oficial no desafio.
'''))

In [ ]:
# ============================================================
# 17. XAI didático: tempo, espectrograma, RPCA, bandas, limiares e saliência
# ============================================================

display(Markdown('''
### Guia para explicar o XAI

XAI significa explicação da inteligência artificial. Em vez de mostrar só um número, esta seção responde seis perguntas simples:

1. **Qual áudio está sendo explicado?**
2. **Por que ele virou candidato a anomalia para os algoritmos?**
3. **Quando aconteceu?**
4. **Em qual frequência aconteceu?**
5. **Foi padrão constante ou impacto raro?**
6. **Quais atributos mais fugiram da normalidade?**
'''))


def choose_xai_example() -> Optional[dict]:
    candidates = []

    if 'BLIND_REPORT' in globals() and isinstance(BLIND_REPORT, pd.DataFrame) and not BLIND_REPORT.empty:
        for model_name in ['GMM + Gamma', 'Mahalanobis + Gamma', 'XGBoost supervisionado', 'Isolation Forest + Gamma']:
            part = BLIND_REPORT[(BLIND_REPORT['modelo'] == model_name) & (BLIND_REPORT['label'] == 1)].copy()
            if not part.empty:
                row = part.sort_values('score_operacional', ascending=False).iloc[0].to_dict()
                row['selected_from'] = 'BLIND_REPORT'
                row['selected_model'] = model_name
                candidates.append(row)

    for frame_name, model_name in [
        ('file_unsup_gmm', 'GMM + Gamma'),
        ('file_unsup', 'Mahalanobis + Gamma'),
        ('file_sup', 'XGBoost supervisionado'),
        ('file_unsup_iforest', 'Isolation Forest + Gamma'),
    ]:
        frame = globals().get(frame_name)
        if isinstance(frame, pd.DataFrame) and not frame.empty and 'label' in frame.columns:
            part = frame[frame['label'] == 1].copy()
            if not part.empty:
                score_col = 'score_operacional' if 'score_operacional' in part.columns else 'score_p90'
                row = part.sort_values(score_col, ascending=False).iloc[0].to_dict()
                row['selected_from'] = frame_name
                row['selected_model'] = model_name
                candidates.append(row)

    if candidates:
        best = sorted(
            candidates,
            key=lambda item: float(item.get('score_operacional', item.get('score_p90', 0.0))),
            reverse=True,
        )[0]
        source_id = int(best['source_id'])
        catalog_row = catalog[catalog['source_id'] == source_id].iloc[0].to_dict()
        catalog_row.update({'selected_from': best.get('selected_from'), 'selected_model': best.get('selected_model')})
        return catalog_row

    anomaly_catalog = catalog[catalog['label'] == 1]
    if not anomaly_catalog.empty:
        row = anomaly_catalog.sample(1, random_state=RANDOM_STATE).iloc[0].to_dict()
        row['selected_from'] = 'catalog'
        row['selected_model'] = 'Catálogo rotulado'
        return row

    if not catalog.empty:
        row = catalog.sample(1, random_state=RANDOM_STATE).iloc[0].to_dict()
        row['selected_from'] = 'catalog'
        row['selected_model'] = 'Exemplo aleatório'
        return row
    return None


def compute_tiny_ast_saliency(ast_image: np.ndarray) -> Optional[dict]:
    if tiny_ast_model is None or not HAS_TORCH or not TINY_AST_ENABLE_SALIENCY:
        return None
    tiny_ast_model.eval()
    batch = torch.tensor(np.asarray(ast_image, dtype=np.float32)[None, None, :, :], dtype=torch.float32, device=tiny_ast_device, requires_grad=True)
    recon, emb = tiny_ast_model(batch, return_embedding=True)
    score = torch.mean((recon - batch) ** 2)
    score.backward()
    grad_map = batch.grad.detach().cpu().numpy()[0, 0]
    saliency = np.abs(grad_map)
    saliency = saliency / (np.max(saliency) + 1e-8)
    time_profile = saliency.mean(axis=0)
    freq_profile = saliency.mean(axis=1)
    peak_t = int(np.argmax(time_profile))
    peak_f = int(np.argmax(freq_profile))
    return {
        'saliency_map': saliency,
        'time_profile': time_profile,
        'freq_profile': freq_profile,
        'peak_time_idx': peak_t,
        'peak_freq_idx': peak_f,
        'peak_value': float(np.max(saliency)),
        'embedding_norm': float(np.linalg.norm(emb.detach().cpu().numpy()[0])),
    }


def classify_physical_band(freq_hz: float) -> str:
    if freq_hz < 800:
        return 'Baixa frequência | vibração/estrutura'
    if freq_hz < 2500:
        return 'Média frequência | atrito/ressonância'
    return 'Alta frequência | ruído agudo/assobio'


def collect_decision_context(source_id: int) -> pd.DataFrame:
    rows = []

    def append_from_df(df: pd.DataFrame, model_name: str, scope: str):
        if not isinstance(df, pd.DataFrame) or df.empty or 'source_id' not in df.columns:
            return
        part = df[df['source_id'] == source_id]
        if part.empty:
            return
        row = part.iloc[0]
        score = float(row['score_operacional']) if 'score_operacional' in row.index else float(row['score_p90'])
        threshold = float(row['limiar']) if 'limiar' in row.index and pd.notna(row['limiar']) else (0.50 if 'supervisionado' in model_name.lower() else np.nan)
        decision = row['predicao'] if 'predicao' in row.index else ('Anômalo' if int(row.get('pred_threshold', row.get('pred_supervised', 0))) == 1 else 'Normal')
        rows.append({
            'Modelo': model_name,
            'Escopo': scope,
            'Score do arquivo': score,
            'Limiar': threshold,
            'Margem sobre limiar': score - threshold if np.isfinite(threshold) else np.nan,
            'Decisão': decision,
        })

    if 'BLIND_REPORT' in globals() and isinstance(BLIND_REPORT, pd.DataFrame) and not BLIND_REPORT.empty:
        for model_name, part in BLIND_REPORT.groupby('modelo'):
            append_from_df(part, model_name, 'Teste cego multi-fonte')

    for scope, df in SUPERVISED_FILE_REPORTS.items() if 'SUPERVISED_FILE_REPORTS' in globals() else []:
        append_from_df(df, 'XGBoost supervisionado', scope)

    unsup_label_map = {
        'externo_domain_shift': 'Mahalanobis + Gamma',
        'externo_domain_shift_gmm': 'GMM + Gamma',
        'externo_domain_shift_iforest': 'Isolation Forest + Gamma',
        'unsup_external_mahalanobis': 'Mahalanobis + Gamma',
        'unsup_external_gmm': 'GMM + Gamma',
        'unsup_external_iforest': 'Isolation Forest + Gamma',
        'teste_cego_multifonte': 'Mahalanobis + Gamma',
        'teste_cego_multifonte_mahalanobis': 'Mahalanobis + Gamma',
        'teste_cego_multifonte_gmm': 'GMM + Gamma',
        'teste_cego_multifonte_iforest': 'Isolation Forest + Gamma',
    }
    for scope, df in UNSUPERVISED_FILE_REPORTS.items() if 'UNSUPERVISED_FILE_REPORTS' in globals() else []:
        append_from_df(df, unsup_label_map.get(scope, scope), scope)

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.drop_duplicates(subset=['Modelo', 'Escopo', 'Score do arquivo'])
        result = result.sort_values(['Margem sobre limiar', 'Score do arquivo'], ascending=False, na_position='last')
    return result


def top_local_feature_deviations(source_windows: pd.DataFrame, chosen_window_id: int, top_k: int = 12) -> pd.DataFrame:
    if source_windows.empty:
        return pd.DataFrame()
    target = source_windows[source_windows['window_id'] == chosen_window_id]
    if target.empty:
        target = source_windows.iloc[[0]]
    target_row = target.iloc[0]

    normal_ref = window_df[(window_df['role_unsup_train']) & (window_df['label'] == 0)]
    if normal_ref.empty:
        return pd.DataFrame()

    ref_mean = normal_ref[FEATURE_COLUMNS].mean()
    ref_std = normal_ref[FEATURE_COLUMNS].std().replace(0, np.nan)
    z = ((target_row[FEATURE_COLUMNS] - ref_mean) / ref_std).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    top_idx = z.abs().sort_values(ascending=False).head(top_k).index.tolist()
    result = pd.DataFrame({
        'Feature': top_idx,
        'Z-score local': z[top_idx].values,
    })

    def family(name: str) -> str:
        if name.startswith('rpca_'):
            return 'Canal A | RPCA'
        if name.startswith('mfcc_') or name.startswith('delta_mfcc_'):
            return 'Canal B | MFCC'
        if name.startswith('mel_band_') or name.startswith('spectral_') or name == 'nmf_reconstruction_error':
            return 'Canal C | Estrutural'
        if name.startswith('ast_emb_') or name == 'tiny_ast_reconstruction_error':
            return 'Canal D | Tiny-AST'
        return 'Outras'

    result['Família'] = result['Feature'].map(family)
    return result


def plot_xai_for_file(row_like):
    row = pd.Series(row_like)
    y, sr = read_signal(row['path'], row['source'])
    y_raw = normalize_signal(y)
    y_clean = champion_dsp(y_raw, sr)
    mel = librosa.feature.melspectrogram(y=y_clean, sr=sr, n_mels=64, n_fft=1024, hop_length=256, power=2.0)
    logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)
    _, rpca_sparse = rpca_decompose_logmel(logmel)
    rpca_sparse_abs = np.abs(rpca_sparse)
    times = librosa.frames_to_time(np.arange(logmel.shape[1]), sr=sr, hop_length=256)
    freqs = librosa.mel_frequencies(n_mels=64, fmin=0, fmax=sr / 2)
    energy_time = rpca_sparse_abs.mean(axis=0)
    if np.allclose(energy_time, 0):
        energy_time = np.mean(np.abs(logmel), axis=0)

    local_threshold = float(np.quantile(energy_time, 0.90))
    peak_candidates, _ = scipy_signal.find_peaks(energy_time, distance=max(1, len(energy_time) // 10))
    if len(peak_candidates) >= 3:
        peak_order = peak_candidates[np.argsort(energy_time[peak_candidates])[-3:]][::-1]
    else:
        peak_order = np.argsort(energy_time)[-3:][::-1]
    peak_order = list(peak_order)

    hotspot_descriptions = [
        'Impacto principal: pico dominante que sustenta a anomalia.',
        'Resposta estrutural: harmônico ou reverberação do mesmo evento.',
        'Eco temporal: recorrência do mecanismo anômalo.',
    ]

    metadata_df = pd.DataFrame([{
        'Arquivo': row['file_name'],
        'Fonte': row['source'],
        'Base': row['dataset'],
        'Máquina': row['machine'],
        'Classe conhecida': row['label_name'],
        'Selecionado a partir de': row.get('selected_from', 'catálogo'),
        'Modelo que puxou o exemplo': row.get('selected_model', 'automático'),
    }])
    display(Markdown('### Áudio escolhido para explicar o XAI'))
    display(metadata_df)

    decision_df = collect_decision_context(int(row['source_id']))
    if not decision_df.empty:
        display(Markdown('### Como os algoritmos reagiram a este arquivo'))
        display(decision_df.round(4))

    fig, axes = plt.subplots(3, 2, figsize=(18, 16))
    fig.suptitle(
        f'XAI do arquivo {row["file_name"]} | {row["source"]} | {row["dataset"]}',
        fontsize=17,
        fontweight='bold',
        y=0.995,
    )

    t = np.arange(len(y_clean)) / sr
    axes[0, 0].plot(t, y_raw, color='gray', alpha=0.65, label='Bruto')
    axes[0, 0].plot(t, y_clean, color='tab:blue', alpha=0.85, label='Limpo por HHT+UKF')
    axes[0, 0].set_title('Waveform do áudio usado no XAI')
    axes[0, 0].set_xlabel('Tempo (s)')
    axes[0, 0].set_ylabel('Amplitude')
    axes[0, 0].legend()

    img = librosa.display.specshow(logmel, x_axis='time', y_axis='mel', sr=sr, hop_length=256, ax=axes[0, 1], cmap='magma')
    axes[0, 1].set_title('Espectrograma Mel do arquivo')
    fig.colorbar(img, ax=axes[0, 1], format='%+2.0f dB')

    img2 = librosa.display.specshow(rpca_sparse_abs, x_axis='time', y_axis='mel', sr=sr, hop_length=256, ax=axes[1, 0], cmap='inferno')
    axes[1, 0].set_title('Mapa RPCA esparso: onde estão os impactos raros?')
    fig.colorbar(img2, ax=axes[1, 0], label='Ativação RPCA')

    axes[1, 1].plot(times, energy_time, color='tab:red', linewidth=2)
    axes[1, 1].axhline(local_threshold, color='black', linestyle='--', linewidth=1.8, label='Limiar local do XAI (p90 do arquivo)')
    axes[1, 1].set_title('Linha temporal dos hotspots RPCA')
    axes[1, 1].set_xlabel('Tempo (s)')
    axes[1, 1].set_ylabel('Intensidade média RPCA')

    hotspot_rows = []
    saliency_lookup = {}

    windows = list(sliding_windows(y_clean, sr))
    source_windows = window_df[window_df['source_id'] == row['source_id']].sort_values('window_id') if 'window_df' in globals() else pd.DataFrame()
    chosen_window_id = 0
    if not source_windows.empty:
        if 'tiny_ast_reconstruction_error' in source_windows.columns:
            chosen_window_id = int(source_windows.sort_values('tiny_ast_reconstruction_error', ascending=False).iloc[0]['window_id'])
        elif 'rpca_sparse_energy' in source_windows.columns:
            chosen_window_id = int(source_windows.sort_values('rpca_sparse_energy', ascending=False).iloc[0]['window_id'])
    chosen_window_id = min(chosen_window_id, max(0, len(windows) - 1))

    saliency_info = None
    if windows:
        ast_image = build_ast_logmel_image(windows[chosen_window_id], sr)
        saliency_info = compute_tiny_ast_saliency(ast_image)
        if saliency_info is not None:
            window_start = chosen_window_id * HOP_SEC
            saliency_times = np.linspace(window_start, window_start + WINDOW_SEC, len(saliency_info['time_profile']))
            for idx, sal_value in enumerate(saliency_info['time_profile']):
                saliency_lookup[float(saliency_times[idx])] = float(sal_value)

    for rank, idx in enumerate(peak_order, start=1):
        label = f'H{rank}'
        peak_time = float(times[idx])
        peak_intensity = float(energy_time[idx])
        if saliency_lookup:
            nearest_time = min(saliency_lookup.keys(), key=lambda val: abs(val - peak_time))
            local_saliency = saliency_lookup[nearest_time]
        else:
            local_saliency = np.nan
        for ax in [axes[0, 1], axes[1, 0]]:
            ax.axvline(peak_time, color='cyan', lw=2, alpha=0.9)
            ax.text(peak_time, 0.95, label, transform=ax.get_xaxis_transform(), color='cyan', fontsize=12, rotation=90, va='top')
        axes[1, 1].scatter([peak_time], [peak_intensity], color='cyan', s=70, zorder=5)
        axes[1, 1].text(peak_time, peak_intensity, f' {label}', color='cyan', fontsize=11, fontweight='bold')
        hotspot_rows.append({
            'Hotspot': label,
            'Tempo (s)': round(peak_time, 3),
            'Intensidade RPCA': round(peak_intensity, 4),
            'Excesso sobre limiar local': round(peak_intensity - local_threshold, 4),
            'Saliência Tiny-AST': round(float(local_saliency), 4) if np.isfinite(local_saliency) else np.nan,
            'Leitura diagnóstica': hotspot_descriptions[min(rank - 1, len(hotspot_descriptions) - 1)],
        })

    axes[1, 1].legend()

    band_edges = [(0, 800, 'Baixa'), (800, 2500, 'Média'), (2500, sr / 2, 'Alta')]
    band_rows = []
    for low, high, name in band_edges:
        mask = (freqs >= low) & (freqs < high)
        activation = float(np.mean(rpca_sparse_abs[mask])) if mask.any() else 0.0
        band_rows.append({'Faixa': name, 'Frequência (Hz)': f'{int(low)} - {int(high)}', 'Ativação média RPCA': activation, 'Leitura física': classify_physical_band((low + high) / 2)})
    band_df = pd.DataFrame(band_rows)
    sns.barplot(data=band_df, y='Faixa', x='Ativação média RPCA', ax=axes[2, 0], palette='Oranges')
    axes[2, 0].set_title('Concentração da anomalia por banda de frequência')
    axes[2, 0].set_xlabel('Ativação média RPCA')
    axes[2, 0].set_ylabel('Banda')

    feature_df = top_local_feature_deviations(source_windows, chosen_window_id, top_k=12)
    if not feature_df.empty:
        sns.barplot(data=feature_df, y='Feature', x='Z-score local', hue='Família', dodge=False, ax=axes[2, 1])
        axes[2, 1].axvline(0, color='black', linewidth=1)
        axes[2, 1].set_title('Features que mais fugiram da normalidade nesta janela')
        axes[2, 1].set_xlabel('Desvio padronizado local (z-score)')
        axes[2, 1].set_ylabel('Feature')
    else:
        axes[2, 1].text(0.5, 0.5, 'Sem referência suficiente para desvio local', ha='center', va='center')
        axes[2, 1].set_axis_off()

    plt.tight_layout(); plt.show()

    hotspot_df = pd.DataFrame(hotspot_rows)
    display(Markdown('### H1, H2 e H3: por que esses pontos chamaram atenção?'))
    display(hotspot_df)
    display(Markdown('### Ativação por bandas'))
    display(band_df)

    if not decision_df.empty:
        best_margin = decision_df['Margem sobre limiar'].dropna()
        if not best_margin.empty:
            margin_text = f'{best_margin.max():.4f}'
        else:
            margin_text = 'sem limiar explícito'
        display(Markdown(f'''
**Por que este arquivo apareceu no XAI?** Porque ele ultrapassou os limites de pelo menos um detector. A tabela acima mostra o `score do arquivo`, o `limiar` e a `margem sobre o limiar`. Em outras palavras: o áudio não entrou no XAI por acaso; ele entrou porque quebrou a região de normalidade aprendida pelos algoritmos. A margem máxima observada nesta seleção foi de **{margin_text}**.
'''))

    if saliency_info is not None and windows:
        ast_freqs = librosa.mel_frequencies(n_mels=AST_IMAGE_MELS, fmin=0, fmax=sr / 2)
        window_start = chosen_window_id * HOP_SEC
        time_axis = np.linspace(window_start, window_start + WINDOW_SEC, AST_IMAGE_TIME_BINS)
        freq_axis = ast_freqs
        peak_time_abs = float(time_axis[saliency_info['peak_time_idx']])
        peak_freq_hz = float(freq_axis[saliency_info['peak_freq_idx']])
        physical_band = classify_physical_band(peak_freq_hz)

        fig, axes = plt.subplots(2, 2, figsize=(16, 11))
        axes[0, 0].imshow(ast_image, aspect='auto', origin='lower', cmap='magma')
        axes[0, 0].set_title(f'Janela AST usada no raio-X | janela {chosen_window_id}')
        axes[0, 0].set_xlabel('Tempo compactado')
        axes[0, 0].set_ylabel('Bandas Mel')

        axes[0, 1].imshow(ast_image, aspect='auto', origin='lower', cmap='gray_r')
        axes[0, 1].imshow(saliency_info['saliency_map'], aspect='auto', origin='lower', cmap='jet', alpha=0.55)
        axes[0, 1].axvline(saliency_info['peak_time_idx'], color='white', lw=2, linestyle='--')
        axes[0, 1].axhline(saliency_info['peak_freq_idx'], color='white', lw=2, linestyle='--')
        axes[0, 1].set_title('Heatmap de saliência por gradiente')
        axes[0, 1].set_xlabel('Tempo compactado')
        axes[0, 1].set_ylabel('Bandas Mel')

        axes[1, 0].plot(time_axis, saliency_info['time_profile'], color='tab:red')
        axes[1, 0].axvline(peak_time_abs, color='black', linestyle='--', lw=2)
        axes[1, 0].set_title('Onde o Tiny-AST mais sentiu dificuldade no tempo')
        axes[1, 0].set_xlabel('Tempo absoluto (s)')
        axes[1, 0].set_ylabel('Saliência média')

        axes[1, 1].plot(freq_axis, saliency_info['freq_profile'], color='tab:purple')
        axes[1, 1].axvline(peak_freq_hz, color='black', linestyle='--', lw=2)
        axes[1, 1].set_title('Em qual faixa de frequência a dificuldade se concentrou')
        axes[1, 1].set_xlabel('Frequência aproximada (Hz)')
        axes[1, 1].set_ylabel('Saliência média')
        plt.tight_layout(); plt.show()

        saliency_df = pd.DataFrame([{
            'Janela analisada': chosen_window_id,
            'Início da janela (s)': round(window_start, 3),
            'Pico de saliência (s)': round(peak_time_abs, 3),
            'Frequência crítica (Hz)': round(peak_freq_hz, 1),
            'Leitura física': physical_band,
            'Intensidade máxima de saliência': round(float(saliency_info['peak_value']), 4),
            'Norma do embedding': round(float(saliency_info['embedding_norm']), 4),
        }])
        display(Markdown('### Raio-X acústico com Tiny-AST auxiliar'))
        display(saliency_df)

    display(Markdown('''
**Resumo para leigos:** o XAI desta seção cruza três ideias ao mesmo tempo:

- **os algoritmos de detecção** mostraram que o arquivo ultrapassou limites de normalidade;
- **o RPCA** mostrou em quais instantes a energia rara explodiu (`H1`, `H2`, `H3`);
- **as features e a saliência** mostraram quais pistas acústicas fizeram isso aparecer como anomalia.

Assim, o diagnóstico deixa de ser “a IA disse que está ruim” e passa a ser “este arquivo, desta base, nesta máquina, rompeu o comportamento saudável por causa destes impactos, nestas bandas e com estas features”.
'''))


xai_row = choose_xai_example()
if xai_row is not None:
    plot_xai_for_file(xai_row)


In [ ]:
# ============================================================
# 17.1. Auditoria da implementação da arquitetura campeã
# ============================================================
# Esta célula confere automaticamente se os pilares descritos na defesa técnica estão presentes.

audit_rows = []


def add_audit(item, status, evidence, note):
    audit_rows.append({'Pilar': item, 'Status': status, 'Evidência': evidence, 'Nota': note})

try:
    train_groups = set(catalog.loc[catalog['role_supervised'].eq('train'), 'group_id'])
    test_groups = set(catalog.loc[catalog['role_supervised'].eq('test_interno'), 'group_id'])
    overlap_sup = train_groups & test_groups
    add_audit('Frente 6 | Blindagem por arquivo', 'OK' if not overlap_sup else 'Falha', f'overlap={len(overlap_sup)}', 'Separação supervisionada sem janelas irmãs cruzadas.')
except Exception as exc:
    add_audit('Frente 6 | Blindagem por arquivo', 'Falha', 'erro', str(exc))

blind_origins = []
if 'catalog' in globals() and 'blind_origin' in catalog.columns:
    blind_origins = sorted([item for item in catalog.loc[catalog['role_blind_test'], 'blind_origin'].dropna().unique().tolist() if item != 'Não se aplica'])

add_audit('Janelas deslizantes', 'OK' if 'sliding_windows' in globals() else 'Falha', f'{WINDOW_SEC:.2f}s | hop={HOP_SEC:.2f}s', 'Segmentação temporal do áudio em trechos menores.')
add_audit('HHT + UKF', 'OK' if ('apply_hht_transient_isolation' in globals() and 'apply_ukf_denoising' in globals()) else 'Falha', 'funções presentes', 'Purificação do sinal antes da inferência.')
add_audit('RPCA', 'OK' if any(c.startswith('rpca_') for c in FEATURE_COLUMNS) else 'Falha', f'{len([c for c in FEATURE_COLUMNS if c.startswith("rpca_")])} features', 'Separa estrutura estável e eventos raros.')
add_audit('Canal A | Impulsivo', 'OK' if {'rpca_sparse_rms', 'rpca_sparse_kurtosis', 'rpca_sparse_energy'}.issubset(set(FEATURE_COLUMNS)) else 'Parcial', 'RPCA + estatísticas', 'Mede agressividade mecânica e impulsividade.')
add_audit('Canal B | Espectral', 'OK' if any(c.startswith('mfcc_') for c in FEATURE_COLUMNS) and any(c.startswith('delta_mfcc_') for c in FEATURE_COLUMNS) else 'Parcial', 'MFCC + deltas', 'Captura timbre e variação dinâmica.')
add_audit('Canal C | Estrutural', 'OK' if 'nmf_reconstruction_error' in FEATURE_COLUMNS else 'Falha', 'NMF reconstruction error', 'Mede quebra da normalidade aprendida.')
add_audit('Tiny-AST embeddings', 'OK' if any(c.startswith('ast_emb_') for c in FEATURE_COLUMNS) else 'Implementado (desligado)' if 'tiny_ast_model' in globals() else 'Falha', f'ativo={ENABLE_TINY_AST_AUX}', 'Embeddings profundos auxiliares sem trocar o campeão leve.')
add_audit('Saliência por gradiente', 'OK' if ('compute_tiny_ast_saliency' in globals() and tiny_ast_model is not None and TINY_AST_ENABLE_SALIENCY) else 'Implementado (desligado)' if 'compute_tiny_ast_saliency' in globals() else 'Falha', f'ativo={TINY_AST_ENABLE_SALIENCY}', 'Heatmap visual tipo raio-X do espectrograma.')
add_audit('Mixup', 'OK' if ('make_mixup_rows' in globals() and MIXUP_ENABLED) else 'Implementado (desligado)' if 'make_mixup_rows' in globals() else 'Falha', f'ativo={MIXUP_ENABLED}', 'Regularização supervisionada por pseudo-anomalias.')
add_audit('Mixup de domínio', 'OK' if ('make_domain_mixup_rows' in globals() and DOMAIN_MIXUP_ENABLED) else 'Implementado (desligado)' if 'make_domain_mixup_rows' in globals() else 'Falha', f'ativo={DOMAIN_MIXUP_ENABLED}', 'Pseudo-normais entre domínios para reduzir sensibilidade a microfone e ambiente.')
add_audit('OE opcional', 'Implementado (desligado)' if (('make_oe_rows' in globals()) and (not OE_ENABLED)) else 'OK' if ('make_oe_rows' in globals() and 'role_oe_train' in window_df.columns) else 'Falha', f'ativo={OE_ENABLED}', 'Outlier Exposure auditado e separado da avaliação.')
add_audit('XGBoost supervisionado', 'OK' if 'supervised_model' in globals() else 'Parcial', 'pipeline supervisionado', 'Campeão supervisionado de borda.')
add_audit('Mahalanobis + Gamma', 'OK' if ('mahalanobis_scores' in globals() and 'mahal_info' in globals()) else 'Falha', 'limiar estatístico', 'Campeão não supervisionado.')
add_audit('GMM + Gamma', 'OK' if ('gmm_model' in globals() and gmm_model is not None and 'gmm_scores' in globals()) else 'Implementado (desligado)' if 'gmm_model' in globals() else 'Falha', f'ativo={ENABLE_GMM_CHALLENGER}', 'Desafiante multimodal para normalidade complexa.')
add_audit('Isolation Forest + Gamma', 'OK' if ('iforest_model' in globals() and iforest_model is not None and 'iforest_scores' in globals()) else 'Implementado (desligado)' if 'iforest_model' in globals() else 'Falha', f'ativo={ENABLE_IFOREST_CHALLENGER}', 'Desafiante baseado em árvores para normalidade caótica.')
add_audit('pAUC@0.1', 'OK' if 'safe_auc' in globals() else 'Falha', 'max_fpr=0.10', 'Avaliação focada na zona de baixo falso alarme.')
add_audit('Teste cego / domain shift', 'OK' if 'BLIND_REPORT' in globals() else 'Parcial', f'origens={", ".join(blind_origins) if blind_origins else "nenhuma"}', 'Validação cega multi-fonte, separada do treino e da avaliação externa padrão.')
add_audit('XAI acústico', 'OK' if 'plot_xai_for_file' in globals() else 'Falha', 'tempo + frequência + RPCA + saliência', 'Explicação visual da decisão.')

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

status_counts = audit_df['Status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Itens']
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=status_counts, x='Status', y='Itens', ax=ax, palette='Set2')
ax.set_title('Auditoria dos pilares da arquitetura campeã', fontsize=14, fontweight='bold')
ax.set_xlabel('Status')
ax.set_ylabel('Quantidade de itens')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout(); plt.show()

display(Markdown('''
**Como usar esta auditoria:**

- `OK`: implementado e encontrado na execução atual.
- `Parcial`: existe no notebook, mas depende de dados ou execução prévia para aparecer completamente.
- `Implementado (desligado)`: a técnica está no pipeline, mas foi mantida desativada por parâmetro.
- `Falha`: o pilar descrito na defesa não foi encontrado e merece correção.
'''))


In [ ]:
# ============================================================
# 18. Predição em um novo arquivo
# ============================================================

def predict_one_file(file_path: str, source: str = 'external') -> pd.DataFrame:
    y, sr = read_signal(file_path, source)
    y = champion_dsp(y, sr)
    rows, mel_rows_local, ast_rows_local = [], [], []
    for win in sliding_windows(y, sr):
        feats, mel_vec, ast_image = extract_base_features(win, sr)
        rows.append(feats)
        mel_rows_local.append(mel_vec)
        ast_rows_local.append(ast_image)
    df = pd.DataFrame(rows)
    if nmf_model is not None and nmf_scaler is not None and mel_rows_local:
        mel_local = np.clip(nmf_scaler.transform(np.vstack(mel_rows_local).astype(np.float32)), 0, None)
        recon = nmf_model.inverse_transform(nmf_model.transform(mel_local))
        df['nmf_reconstruction_error'] = np.mean((mel_local - recon) ** 2, axis=1)
    else:
        df['nmf_reconstruction_error'] = 0.0

    if tiny_ast_model is not None and ast_rows_local:
        ast_local = np.stack(ast_rows_local).astype(np.float32)
        ast_embeddings_local, ast_recon_local = compute_tiny_ast_features(ast_local)
        if ast_embeddings_local.size:
            for i in range(ast_embeddings_local.shape[1]):
                df[f'ast_emb_{i+1:02d}'] = ast_embeddings_local[:, i]
        df['tiny_ast_reconstruction_error'] = ast_recon_local
    elif 'tiny_ast_reconstruction_error' in FEATURE_COLUMNS:
        df['tiny_ast_reconstruction_error'] = 0.0

    for col in FEATURE_COLUMNS:
        if col not in df.columns:
            df[col] = 0.0
    result = {'arquivo': Path(file_path).name, 'janelas': len(df)}
    if supervised_model is not None:
        scores = supervised_model.predict_proba(df[FEATURE_COLUMNS])[:, 1] if hasattr(supervised_model.named_steps['clf'], 'predict_proba') else supervised_model.decision_function(df[FEATURE_COLUMNS])
        result['score_supervisionado_p90'] = float(np.quantile(scores, 0.90))
        result['decisao_supervisionada'] = 'Anômalo' if result['score_supervisionado_p90'] >= 0.50 else 'Normal'
    scores_u = mahalanobis_scores(df[FEATURE_COLUMNS])
    result['score_mahalanobis_p90'] = float(np.quantile(scores_u, 0.90))
    result['threshold_mahalanobis'] = float(mahal_info['threshold_final'])
    result['decisao_nao_supervisionada'] = 'Anômalo' if result['score_mahalanobis_p90'] >= mahal_info['threshold_final'] else 'Normal'
    if gmm_model is not None and 'gmm_scores' in globals():
        scores_g = gmm_scores(df[FEATURE_COLUMNS])
        result['score_gmm_p90'] = float(np.quantile(scores_g, 0.90))
        result['threshold_gmm'] = float(gmm_threshold_final)
        result['decisao_gmm'] = 'Anômalo' if result['score_gmm_p90'] >= gmm_threshold_final else 'Normal'
    if iforest_model is not None and 'iforest_scores' in globals():
        scores_i = iforest_scores(df[FEATURE_COLUMNS])
        result['score_iforest_p90'] = float(np.quantile(scores_i, 0.90))
        result['threshold_iforest'] = float(iforest_threshold_final)
        result['decisao_iforest'] = 'Anômalo' if result['score_iforest_p90'] >= iforest_threshold_final else 'Normal'
    if 'tiny_ast_reconstruction_error' in df.columns:
        result['score_tiny_ast_recon_p90'] = float(np.quantile(df['tiny_ast_reconstruction_error'], 0.90))
    return pd.DataFrame([result])

# Exemplo:
# predict_one_file('/caminho/para/audio.wav')


In [ ]:
# ============================================================
# 19. Salvamento dos artefatos
# ============================================================

tiny_ast_serialized_path = None
if HAS_TORCH and tiny_ast_model is not None:
    tiny_ast_serialized_path = ARTIFACT_DIR / 'tiny_ast_aux.pt'
    model_snapshot = {
        'state_dict': tiny_ast_model.cpu().state_dict(),
        'config': {
            'num_mels': AST_IMAGE_MELS,
            'time_steps': AST_IMAGE_TIME_BINS,
            'embed_dim': TINY_AST_EMBED_DIM,
            'num_heads': TINY_AST_HEADS,
            'num_layers': TINY_AST_LAYERS,
        },
    }
    torch.save(model_snapshot, tiny_ast_serialized_path)
    if tiny_ast_device is not None:
        tiny_ast_model.to(tiny_ast_device)
    if not tiny_ast_history.empty:
        tiny_ast_history.to_csv(ARTIFACT_DIR / 'tiny_ast_aux_history.csv', index=False)

artifacts = {
    'feature_columns': FEATURE_COLUMNS,
    'sr_target': SR_TARGET,
    'window_sec': WINDOW_SEC,
    'hop_sec': HOP_SEC,
    'fpr_target': FPR_TARGET,
    'rpca_enabled': bool(ENABLE_RPCA),
    'rpca_max_iter': int(RPCA_MAX_ITER),
    'rpca_tol': float(RPCA_TOL),
    'mixup_enabled': bool(MIXUP_ENABLED),
    'mixup_ratio': float(MIXUP_RATIO),
    'mixup_use_fixed_lambda': bool(MIXUP_USE_FIXED_LAMBDA),
    'mixup_lambda': float(MIXUP_LAMBDA),
    'mixup_alpha': float(MIXUP_ALPHA),
    'domain_mixup_enabled': bool(DOMAIN_MIXUP_ENABLED),
    'domain_mixup_ratio': float(DOMAIN_MIXUP_RATIO),
    'domain_mixup_sample_weight': float(DOMAIN_MIXUP_SAMPLE_WEIGHT),
    'oe_enabled': bool(OE_ENABLED),
    'oe_max_windows': int(OE_MAX_WINDOWS),
    'hyperparam_search_enabled': bool(ENABLE_HYPERPARAM_SEARCH),
    'final_threshold_mahalanobis': float(mahal_info['threshold_final']),
    'gmm_enabled': bool(ENABLE_GMM_CHALLENGER),
    'gmm_best_components': None if gmm_best_components is None else int(gmm_best_components),
    'gmm_covariance_type': GMM_COVARIANCE_TYPE,
    'gmm_final_threshold': None if not np.isfinite(gmm_threshold_final) else float(gmm_threshold_final),
    'iforest_enabled': bool(ENABLE_IFOREST_CHALLENGER),
    'iforest_trees': int(IFOREST_N_ESTIMATORS),
    'iforest_final_threshold': None if not np.isfinite(iforest_threshold_final) else float(iforest_threshold_final),
    'tiny_ast_enabled': bool(ENABLE_TINY_AST_AUX),
    'tiny_ast_trained': bool(tiny_ast_model is not None),
    'tiny_ast_feature_columns': [c for c in FEATURE_COLUMNS if c.startswith('ast_emb_') or c == 'tiny_ast_reconstruction_error'],
    'tiny_ast_model_path': None if tiny_ast_serialized_path is None else str(tiny_ast_serialized_path),
    'gamma_params': {
        'shape': None if not np.isfinite(mahal_info['shape']) else float(mahal_info['shape']),
        'loc': float(mahal_info['loc']),
        'scale': None if not np.isfinite(mahal_info['scale']) else float(mahal_info['scale']),
    },
}
joblib.dump({
    'supervised_model': supervised_model,
    'unsup_imputer': unsup_imputer,
    'unsup_scaler': unsup_scaler,
    'cov_model': cov_model,
    'gmm_model': gmm_model,
    'iforest_model': iforest_model,
    'nmf_model': nmf_model,
    'nmf_scaler': nmf_scaler,
    'metadata': artifacts,
}, ARTIFACT_DIR / 'arquitetura_campea_modelos.joblib')
with open(ARTIFACT_DIR / 'arquitetura_campea_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(artifacts, f, indent=2, ensure_ascii=False)
print('Artefatos salvos em:', ARTIFACT_DIR)


## Checklist final para explicar o resultado

- **Janelas deslizantes:** o áudio é dividido em pequenos pedaços para capturar eventos curtos, como batidas e atritos.
- **HHT + UKF:** limpa o sinal e destaca transientes mecânicos relevantes.
- **RPCA:** separa o padrão repetitivo da máquina da parte rara/esparsa, que costuma carregar a anomalia.
- **MFCC/Mel/estatísticas:** transformam som em números interpretáveis: timbre, energia, impulsividade, rugosidade e distribuição espectral.
- **NMF:** mede quanto o som foge da receita de normalidade aprendida.
- **Tiny-AST auxiliar:** não substitui o campeão leve; injeta embeddings profundos e gera um heatmap de saliência mais visual.
- **Mixup:** cria exemplos intermediários para o supervisionado aprender fronteiras mais robustas.
- **Mixup de domínio:** mistura normais de fontes diferentes para reduzir susto com microfone, ambiente e máquina novos.
- **OE opcional:** usa sons auxiliares como exemplos de fora do domínio, com peso menor para não distorcer a falha real.
- **XGBoost supervisionado:** use quando houver exemplos rotulados de falha.
- **Mahalanobis + Gamma:** use quando a realidade industrial só fornece sons normais.
- **GMM + Gamma:** use como desafiante quando a máquina saudável parece ter mais de um comportamento normal.
- **Isolation Forest + Gamma:** use como desafiante quando a normalidade é caótica e pouco gaussiana.
- **pAUC@0.1:** mede desempenho na zona de baixo falso alarme, a região mais importante para chão de fábrica.
- **XAI:** mostra o momento, a frequência, a banda, a componente esparsa e o heatmap de saliência que motivaram o alerta.
- **Teste cego multi-fonte:** compara Drive, Kaggle cego e DCASE cego sem reutilizar arquivos marcados para treino.

Em linguagem simples: o sistema aprende como uma máquina saudável costuma soar. Depois transforma cada novo áudio em números. Se esses números ficam longe da zona normal, o sistema dispara um alerta e mostra onde esse desvio apareceu. A Frente 11 era a arena de busca; esta Arquitetura Campeã virou o produto enxuto, mas agora resgatando os blocos que ajudam a enfrentar melhor o `domain shift` sem perder a prontidão para Edge.
